# Setup

Run this cell first to install every dependency declared in [pyproject.toml](pyproject.toml) into the active notebook environment. The installer reads the dependency list directly from the project configuration and also reuses any additional package indexes declared there.

If the cell installs new packages successfully, restart the kernel before running the rest of the notebook.

In [ ]:
import subprocess
import sys
from pathlib import Path
import tomllib


def find_repo_root(start_path: Path) -> Path:
    for candidate in (start_path, *start_path.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate pyproject.toml from the current working directory.")


def ensure_pip_available() -> None:
    pip_check_command = [sys.executable, "-m", "pip", "--version"]
    pip_check_result = subprocess.run(
        pip_check_command,
        capture_output=True,
        text=True,
    )
    if pip_check_result.returncode == 0:
        return

    print("pip is not available in this interpreter. Bootstrapping pip with ensurepip...")
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

    pip_check_result = subprocess.run(
        pip_check_command,
        capture_output=True,
        text=True,
    )
    if pip_check_result.returncode != 0:
        details = pip_check_result.stderr.strip() or pip_check_result.stdout.strip()
        raise RuntimeError(
            "pip is still unavailable after ensurepip completed. "
            f"Interpreter: {sys.executable}. Details: {details}"
        )


repo_root = find_repo_root(Path.cwd().resolve())
pyproject_path = repo_root / "pyproject.toml"
pyproject = tomllib.loads(pyproject_path.read_text(encoding="utf-8"))

project_config = dict(pyproject.get("project", {}))
dependencies = list(project_config.get("dependencies", []))
if not dependencies:
    raise ValueError(f"No project.dependencies entries were found in {pyproject_path}")

uv_config = dict(pyproject.get("tool", {}).get("uv", {}))
extra_index_urls = list(
    dict.fromkeys(
        entry["url"]
        for entry in uv_config.get("index", [])
        if isinstance(entry, dict) and entry.get("url")
    )
)

ensure_pip_available()

install_command = [sys.executable, "-m", "pip", "install", "--upgrade"]
for index_url in extra_index_urls:
    install_command.extend(["--extra-index-url", index_url])
install_command.extend(dependencies)

print(f"Installing {len(dependencies)} dependencies from {pyproject_path.name}.")
print(f"Python executable: {sys.executable}")
if project_config.get("requires-python"):
    print(f"Project requires Python {project_config['requires-python']}")
if extra_index_urls:
    print("Using additional package indexes:")
    for index_url in extra_index_urls:
        print(f"- {index_url}")

subprocess.run(install_command, check=True, cwd=repo_root)

print("Dependency installation complete.")
print("Restart the kernel if any newly installed packages are not immediately available.")

Installing 17 dependencies from pyproject.toml.
Python executable: c:\Users\eselerio\projects\pibre-model\.venv\Scripts\python.exe
Project requires Python >=3.11,<3.13
Using additional package indexes:
- https://download.pytorch.org/whl/cu124


In [1]:
from src.utils.simulation import load_ml_orchestration_params

analysis_metric = "projected_MSE"

benchmark_orchestration_params = load_ml_orchestration_params()
benchmark_settings = dict(benchmark_orchestration_params.get("benchmark", {}))
USE_OPTUNA = bool(benchmark_settings.get("use_optuna", False))

print(f"Classical benchmark Optuna enabled: {USE_OPTUNA}")
print(f"Analysis metric set to: {analysis_metric}")

Classical benchmark Optuna enabled: False
Analysis metric set to: projected_MSE


# Main Orchestration

This notebook runs the ASM2D-TSN simulation model from the source package and persists the generated dataset and metadata under the repository data contract.

In [2]:
import json

import numpy as np
import pandas as pd
from IPython.display import display as ipython_display

from src.models.simulation.asm2d_tsn_simulation import get_asm2d_tsn_matrices, load_asm2d_tsn_simulation_params
from src.utils.simulation import get_repo_root

repo_root = get_repo_root()
simulation_dir = repo_root / "data" / "asm2d-tsn" / "simulation"

dataset_candidates = {
    path.stem.removeprefix("data_"): path
    for path in simulation_dir.glob("data_*.csv")
}
metadata_candidates = {
    path.stem.removeprefix("metadata_"): path
    for path in simulation_dir.glob("metadata_*.json")
}

if not dataset_candidates:
    raise FileNotFoundError(f"No ASM2D-TSN datasets found in {simulation_dir}")
if not metadata_candidates:
    raise FileNotFoundError(f"No ASM2D-TSN metadata files found in {simulation_dir}")

shared_timestamps = sorted(set(dataset_candidates) & set(metadata_candidates))
if shared_timestamps:
    latest_timestamp = shared_timestamps[-1]
    latest_dataset_path = dataset_candidates[latest_timestamp]
    latest_metadata_path = metadata_candidates[latest_timestamp]
else:
    latest_dataset_path = max(dataset_candidates.values(), key=lambda path: path.stat().st_mtime)
    latest_metadata_path = max(metadata_candidates.values(), key=lambda path: path.stat().st_mtime)

dataset = pd.read_csv(latest_dataset_path)
metadata = json.loads(latest_metadata_path.read_text(encoding="utf-8-sig"))
artifact_paths = {
    "dataset_csv": latest_dataset_path,
    "metadata_json": latest_metadata_path,
}

model_params = load_asm2d_tsn_simulation_params(repo_root)
matrix_bundle = get_asm2d_tsn_matrices(model_params, repo_root=repo_root)
state_columns = list(matrix_bundle["state_columns"])
measured_output_columns = list(matrix_bundle["measured_output_columns"])
petersen_matrix = np.asarray(matrix_bundle["petersen_matrix"], dtype=float)
composition_matrix = np.asarray(matrix_bundle["composition_matrix"], dtype=float)

metadata_state_columns = list(metadata.get("state_columns", []))
metadata_measured_output_columns = list(metadata.get("measured_output_columns", []))
if metadata_state_columns and metadata_state_columns != state_columns:
    raise ValueError(
        "Metadata state_columns do not match workbook-derived state columns. Regenerate simulation artifacts."
    )
if metadata_measured_output_columns and metadata_measured_output_columns != measured_output_columns:
    raise ValueError(
        "Metadata measured_output_columns do not match workbook composition_matrix columns. Regenerate simulation artifacts."
    )

metadata_table = pd.DataFrame(
    {
        "field": list(metadata.keys()),
        "value": [
            json.dumps(value)
            if isinstance(value, (dict, list))
            else None
            if value is None
            else str(value)
            for value in metadata.values()
        ],
    }
)

matrix_source = matrix_bundle["composition_workbook_path"]
print(f"Loaded {len(dataset)} rows for {metadata['simulation_name']}.")
print(f"Dataset loaded from: {artifact_paths['dataset_csv']}")
print(f"Metadata loaded from: {artifact_paths['metadata_json']}")
print(f"Matrix source: {matrix_source}")
print(f"Composition cache source: {matrix_bundle['composition_cache_source']}")
print(f"Petersen matrix shape: {petersen_matrix.shape}")
print(f"Composition matrix shape: {composition_matrix.shape}")

ipython_display(dataset.head())
ipython_display(metadata_table)
ipython_display(pd.DataFrame(petersen_matrix, index=metadata["processes"], columns=state_columns))
ipython_display(pd.DataFrame(composition_matrix, index=measured_output_columns, columns=state_columns))

Loaded 10000 rows for asm2d_tsn_simulation.
Dataset loaded from: C:\Users\eselerio\projects\pibre-model\data\asm2d-tsn\simulation\data_20260419_092021.csv
Metadata loaded from: C:\Users\eselerio\projects\pibre-model\data\asm2d-tsn\simulation\metadata_20260419_092021.json
Matrix source: C:/Users/eselerio/projects/pibre-model/data/asm2d-tsn/asm2d_tsn_workbook.xlsx
Composition cache source: cache
Petersen matrix shape: (28, 20)
Composition matrix shape: (4, 20)


,HRT,Aeration,In_S_O,In_S_F,In_S_A,In_S_NH4,In_S_NO2,In_S_NO3,In_S_N2,In_S_PO4,...,Out_X_PP,Out_X_PHA,Out_X_AOB,Out_X_NOB,Out_X_MeP,Out_X_MeOH,Out_COD,Out_TN,Out_TP,Out_TSS
0,33.304536,0.769557,0.096633,24.856112,45.037878,20.205008,0.847323,2.750122,1.331549,13.602139,...,8.055621,15.788761,3.544594,7.571050e+00,12.514621,1.178597,329.364397,29.874916,22.231357,256.386263
1,34.737404,1.781444,0.356728,105.884933,19.757144,31.459997,0.815541,7.123836,1.872135,11.845098,...,8.685990,0.269811,4.375561,1.000000e-08,14.786741,3.553763,352.745905,42.958758,19.903832,308.226114
2,29.031310,2.152360,0.250623,143.366387,59.712617,19.771197,1.115366,3.726128,1.391105,8.036099,...,19.759180,27.693071,6.361204,3.225982e+00,9.407499,2.508384,542.779459,35.567396,31.584582,478.060957
3,34.288016,2.132976,0.185229,65.572587,32.831810,31.426996,2.506980,2.012990,1.439719,14.511533,...,14.236173,1.996190,6.712771,6.522642e+00,5.184674,3.404109,423.805900,47.603582,22.795625,397.746881
4,7.399591,2.085684,0.189963,171.106295,75.317435,35.886877,1.503144,0.192062,1.961602,15.455573,...,7.397960,23.745535,5.592335,7.199851e+00,20.011720,1.053707,473.412175,39.901603,34.102376,349.873925


,field,value
0,simulation_name,asm2d_tsn_simulation
1,n_samples,10000
2,random_seed,42
3,sampling_method,latin_hypercube
4,dependent_columns,"[""Out_COD"", ""Out_TN"", ""Out_TP"", ""Out_TSS""]"
5,independent_columns,"[""HRT"", ""Aeration"", ""In_S_O"", ""In_S_F"", ""In_S_..."
6,identifier_columns,[]
7,ignored_columns,"[""In_COD"", ""In_TN"", ""In_TP"", ""In_TSS"", ""Out_S_..."
8,dataset_file,data/asm2d-tsn/simulation/data_20260419_092021...
9,state_columns,"[""S_O"", ""S_F"", ""S_A"", ""S_NH4"", ""S_NO2"", ""S_NO3..."


,S_O,S_F,S_A,S_NH4,S_NO2,S_NO3,S_N2,S_PO4,S_I,S_ALK,X_I,X_S,X_H,X_PAO,X_PP,X_PHA,X_AOB,X_NOB,X_MeP,X_MeOH
Aerobic hydrolysis,0.000000,1.0,0.0,-0.000000,0.000000,0.000,0.000000,-0.000,0.0,-0.000000,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Anoxic hydrolysis (NO2),0.000000,1.0,0.0,-0.000000,0.000000,0.000,0.000000,-0.000,0.0,-0.000000,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Anoxic hydrolysis (NO3),0.000000,1.0,0.0,-0.000000,0.000000,0.000,0.000000,-0.000,0.0,-0.000000,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Anaerobic hydrolysis,0.000000,1.0,0.0,-0.000000,0.000000,0.000,0.000000,-0.000,0.0,-0.000000,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Aerobic growth on S_F,-0.600000,-1.6,0.0,-0.070000,0.000000,0.000,0.000000,-0.020,0.0,-0.005645,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Aerobic growth on S_A,-0.600000,0.0,-1.6,-0.070000,0.000000,0.000,0.000000,-0.020,0.0,-0.005645,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Anoxic growth on S_F (NO3 -> NO2),0.000000,-1.6,0.0,-0.070000,0.525000,-0.525,0.000000,-0.020,0.0,-0.005645,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Anoxic growth on S_F (NO2 -> N2),0.000000,-1.6,0.0,-0.070000,-0.348837,0.000,0.348837,-0.020,0.0,0.019272,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Anoxic growth on S_A (NO3 -> NO2),0.000000,0.0,-1.6,-0.070000,0.525000,-0.525,0.000000,-0.020,0.0,-0.005645,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
Anoxic growth on S_A (NO2 -> N2),0.000000,0.0,-1.6,-0.070000,-0.348837,0.000,0.348837,-0.020,0.0,0.019272,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00


,S_O,S_F,S_A,S_NH4,S_NO2,S_NO3,S_N2,S_PO4,S_I,S_ALK,X_I,X_S,X_H,X_PAO,X_PP,X_PHA,X_AOB,X_NOB,X_MeP,X_MeOH
COD,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.00,0.0,1.00,1.00,1.00,1.00,0.00,1.0,1.00,1.00,0.000,0.0
TN,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.01,0.0,0.02,0.00,0.07,0.07,0.00,0.0,0.07,0.07,0.000,0.0
TP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.00,0.0,0.01,0.00,0.02,0.02,1.00,0.0,0.02,0.02,0.205,0.0
TSS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.75,0.75,0.90,0.90,3.23,0.6,0.90,0.90,1.000,1.0


In [3]:
import numpy as np
import pandas as pd
from IPython.display import display as ipython_display
from scipy.linalg import null_space

from src.utils.process import has_active_projection

icsor_constraint_basis = null_space(petersen_matrix)
icsor_A_matrix = icsor_constraint_basis.T

icsor_A_matrix = np.round(icsor_A_matrix, 5)
icsor_A_matrix[np.abs(icsor_A_matrix) < 1e-10] = 0.0

for row_index in range(icsor_A_matrix.shape[0]):
    non_zero_entries = icsor_A_matrix[row_index, icsor_A_matrix[row_index, :] != 0]
    if len(non_zero_entries) > 0:
        icsor_A_matrix[row_index, :] = icsor_A_matrix[row_index, :] / non_zero_entries[0]

A_matrix = icsor_A_matrix.copy()

classical_projection_active = has_active_projection(A_matrix)
fractional_constraint_dimension = int(A_matrix.shape[1])
fractional_nullity = int(A_matrix.shape[0])
fractional_stoichiometric_rank = int(np.linalg.matrix_rank(petersen_matrix))
classical_projection_status = pd.DataFrame(
    [
        {
            "projection_active": classical_projection_active,
            "constraint_status": "active" if classical_projection_active else "inactive_trivial_null_space",
            "fractional_constraint_dimension": fractional_constraint_dimension,
            "rank_S_fractional": fractional_stoichiometric_rank,
            "nullity_S_fractional": fractional_nullity,
        }
    ]
)

print(f"Fractional Petersen matrix shape: {petersen_matrix.shape}")
print(f"icsor invariant matrix shape: {icsor_A_matrix.shape}")
print(f"Shared fractional-space invariant matrix shape kept for downstream regressors: {A_matrix.shape}")
if classical_projection_active:
    print("Classical fractional-space projection remains active because the Petersen null space is non-trivial.")
else:
    print("Classical fractional-space projection is inactive because null_space(nu) is trivial in the ASM component basis.")
    print("Projected classical results and fractional-space discrepancy summaries are suppressed downstream.")

ipython_display(
    pd.DataFrame(
        icsor_A_matrix,
        index=[f"constraint_{index + 1}" for index in range(icsor_A_matrix.shape[0])],
        columns=metadata["state_columns"],
    )
)
ipython_display(
    pd.DataFrame(
        A_matrix,
        index=[f"constraint_{index + 1}" for index in range(A_matrix.shape[0])],
        columns=metadata["state_columns"],
    )
)
ipython_display(classical_projection_status)

Fractional Petersen matrix shape: (28, 20)
icsor invariant matrix shape: (5, 20)
Shared fractional-space invariant matrix shape kept for downstream regressors: (5, 20)
Classical fractional-space projection remains active because the Petersen null space is non-trivial.


,S_O,S_F,S_A,S_NH4,S_NO2,S_NO3,S_N2,S_PO4,S_I,S_ALK,X_I,X_S,X_H,X_PAO,X_PP,X_PHA,X_AOB,X_NOB,X_MeP,X_MeOH
constraint_1,-0.0,-0.0,-0.0,1.0,1.011634,1.011634,1.005817,0.032469,-0.064208,0.081516,0.020470,-0.0,0.071119,0.071119,0.035103,-0.0,0.071119,0.071119,-0.067167,-0.104986
constraint_2,-0.0,-0.0,-0.0,1.0,0.271313,0.271313,0.635657,-20.448101,3.408615,-5.101107,-0.193539,-0.0,-0.367634,-0.367634,-20.612623,-0.0,-0.367634,-0.367634,-2.646425,2.239306
constraint_3,-0.0,-0.0,-0.0,1.0,-2.442802,-2.442802,-0.721080,-1.190231,-52.883033,-24.098972,-0.034062,-0.0,-0.089974,-0.089974,-1.967866,-0.0,-0.089974,-0.089974,-15.929949,-21.916452
constraint_4,0.0,0.0,0.0,1.0,-0.179395,-0.179395,0.410256,1.386737,2.384062,-8.255580,0.019369,0.0,0.051098,0.051098,1.120365,0.0,0.051098,0.051098,1.657904,2.015495
constraint_5,-0.0,-0.0,-0.0,1.0,3.373372,3.373372,2.186686,-7.149059,-71.205499,16.616498,-0.021708,-0.0,0.020260,0.020260,-6.612156,-0.0,0.020260,0.020260,71.000000,102.138929


,S_O,S_F,S_A,S_NH4,S_NO2,S_NO3,S_N2,S_PO4,S_I,S_ALK,X_I,X_S,X_H,X_PAO,X_PP,X_PHA,X_AOB,X_NOB,X_MeP,X_MeOH
constraint_1,-0.0,-0.0,-0.0,1.0,1.011634,1.011634,1.005817,0.032469,-0.064208,0.081516,0.020470,-0.0,0.071119,0.071119,0.035103,-0.0,0.071119,0.071119,-0.067167,-0.104986
constraint_2,-0.0,-0.0,-0.0,1.0,0.271313,0.271313,0.635657,-20.448101,3.408615,-5.101107,-0.193539,-0.0,-0.367634,-0.367634,-20.612623,-0.0,-0.367634,-0.367634,-2.646425,2.239306
constraint_3,-0.0,-0.0,-0.0,1.0,-2.442802,-2.442802,-0.721080,-1.190231,-52.883033,-24.098972,-0.034062,-0.0,-0.089974,-0.089974,-1.967866,-0.0,-0.089974,-0.089974,-15.929949,-21.916452
constraint_4,0.0,0.0,0.0,1.0,-0.179395,-0.179395,0.410256,1.386737,2.384062,-8.255580,0.019369,0.0,0.051098,0.051098,1.120365,0.0,0.051098,0.051098,1.657904,2.015495
constraint_5,-0.0,-0.0,-0.0,1.0,3.373372,3.373372,2.186686,-7.149059,-71.205499,16.616498,-0.021708,-0.0,0.020260,0.020260,-6.612156,-0.0,0.020260,0.020260,71.000000,102.138929


,projection_active,constraint_status,fractional_constraint_dimension,rank_S_fractional,nullity_S_fractional
0,True,active,20,15,5


In [4]:
import pandas as pd

from src.utils.analysis import build_notebook_table_recorder
from src.utils.process import (
    apply_train_test_split_indices,
    build_fractional_input_fractional_output_dataset,
    build_icsor_supervised_dataset,
    make_train_test_split_indices,
    sample_dataset_split_indices,
    select_dataset_rows,
)
from src.utils.simulation import load_ml_orchestration_params, make_simulation_timestamp

ml_orchestration_params = load_ml_orchestration_params()
ml_orchestration = dict(ml_orchestration_params["hyperparameters"])
benchmark_settings = dict(ml_orchestration_params.get("benchmark", {}))
USE_OPTUNA = bool(benchmark_settings.get("use_optuna", False))

classical_benchmark_dataset = build_fractional_input_fractional_output_dataset(
    dataset,
    metadata,
    composition_matrix,
)
icsor_dataset = build_icsor_supervised_dataset(
    dataset,
    metadata,
    composition_matrix,
)

shared_split_indices = make_train_test_split_indices(
    dataset.index,
    test_fraction=float(ml_orchestration["test_fraction"]),
    random_seed=int(ml_orchestration["random_seed"]),
)
main_dataset_splits = apply_train_test_split_indices(
    classical_benchmark_dataset,
    shared_split_indices,
)
icsor_dataset_splits = apply_train_test_split_indices(
    icsor_dataset,
    shared_split_indices,
)

optuna_indices = sample_dataset_split_indices(
    shared_split_indices.train,
    fraction=float(ml_orchestration["optuna_dataset_fraction"]),
    random_seed=int(ml_orchestration["random_seed"]),
)
optuna_dataset = select_dataset_rows(
    classical_benchmark_dataset,
    optuna_indices,
)
tuning_split_indices = make_train_test_split_indices(
    optuna_indices,
    test_fraction=float(ml_orchestration["optuna_test_fraction"]),
    random_seed=int(ml_orchestration["random_seed"]),
)
tuning_dataset_splits = apply_train_test_split_indices(
    classical_benchmark_dataset,
    tuning_split_indices,
)
icsor_tuning_dataset_splits = apply_train_test_split_indices(
    icsor_dataset,
    tuning_split_indices,
)

split_sizes = {
    "train": len(main_dataset_splits.train.features),
    "test": len(main_dataset_splits.test.features),
    "optuna_dataset": len(optuna_dataset.features),
    "optuna_train": len(tuning_dataset_splits.train.features),
    "optuna_test": len(tuning_dataset_splits.test.features),
}
classical_split_sizes = {
    "train": len(main_dataset_splits.train.features),
    "test": len(main_dataset_splits.test.features),
    "fractional_input_features": len(classical_benchmark_dataset.features.columns),
    "fractional_output_targets": len(classical_benchmark_dataset.targets.columns),
    "fractional_constraints": len(classical_benchmark_dataset.constraint_reference.columns),
}
icsor_split_sizes = {
    "train": len(icsor_dataset_splits.train.features),
    "test": len(icsor_dataset_splits.test.features),
    "optuna_train": len(icsor_tuning_dataset_splits.train.features),
    "optuna_test": len(icsor_tuning_dataset_splits.test.features),
    "fractional_features": len(icsor_dataset.features.columns),
    "fractional_targets": len(icsor_dataset.targets.columns),
    "fractional_constraints": len(icsor_dataset.constraint_reference.columns),
}
split_alignment_ok = (
    main_dataset_splits.train.features.index.equals(icsor_dataset_splits.train.features.index)
    and main_dataset_splits.test.features.index.equals(icsor_dataset_splits.test.features.index)
)
feature_alignment_ok = (
    main_dataset_splits.train.features.columns.equals(icsor_dataset_splits.train.features.columns)
    and main_dataset_splits.test.features.columns.equals(icsor_dataset_splits.test.features.columns)
)
target_alignment_ok = (
    main_dataset_splits.train.targets.columns.equals(icsor_dataset_splits.train.targets.columns)
    and main_dataset_splits.test.targets.columns.equals(icsor_dataset_splits.test.targets.columns)
)
optuna_subset_ok = set(shared_split_indices.test).isdisjoint(optuna_indices)
icsor_optuna_alignment_ok = (
    tuning_dataset_splits.train.features.index.equals(icsor_tuning_dataset_splits.train.features.index)
    and tuning_dataset_splits.test.features.index.equals(icsor_tuning_dataset_splits.test.features.index)
)

benchmark_contract_summary = pd.DataFrame(
    [
        {
            "classical_feature_space": "operational_plus_fractional_influent",
            "icsor_feature_space": "operational_plus_fractional_influent",
            "shared_target_space": "fractional_effluent_components",
            "direct_comparison_space": "externally_collapsed_measured_outputs",
            "shared_split_indices": split_alignment_ok,
            "shared_feature_columns": feature_alignment_ok,
            "shared_target_columns": target_alignment_ok,
            "optuna_subset_from_training_pool_only": optuna_subset_ok,
            "classical_optuna_enabled": USE_OPTUNA,
            "icsor_optuna_split_aligned": icsor_optuna_alignment_ok,
        }
    ]
)

ml_setup_timestamp = make_simulation_timestamp()
record_setup_table = build_notebook_table_recorder(
    "orchestration/shared_setup",
    repo_root=repo_root,
    timestamp=ml_setup_timestamp,
    index=False,
)

print("Notebook-managed ML orchestration is ready.")
print(
    "Classical regressors and icsor now share the same operational-plus-fractional input basis and the same fractional effluent target basis for native training."
)
print(
    "Measured-output comparison remains available downstream through the external composition-matrix collapse layer."
)
print(f"Shared split indices aligned across icsor and the classical benchmark: {split_alignment_ok}")
print(f"Shared feature columns aligned across icsor and the classical benchmark: {feature_alignment_ok}")
print(f"Shared target columns aligned across icsor and the classical benchmark: {target_alignment_ok}")
print(f"Optuna subset drawn only from the shared training pool: {optuna_subset_ok}")
print(f"ICSOR tuning split indices aligned with the notebook-owned Optuna split: {icsor_optuna_alignment_ok}")
print(f"Saved orchestration tables under timestamp: {ml_setup_timestamp}")

record_setup_table(
    "ML orchestration settings",
    "This table shows the shared orchestration hyperparameters and benchmark toggles that control the train-test split, the optional Optuna subset, and whether tuned classical runs are enabled.",
    pd.DataFrame([{**ml_orchestration, **benchmark_settings}]),
)
record_setup_table(
    "Classical benchmark split summary",
    "This table summarizes the split sizes and dataset dimensions used by the classical benchmark models after aligning their native inputs and native targets with icsor.",
    pd.DataFrame([classical_split_sizes]),
)
record_setup_table(
    "icsor split summary",
    "This table summarizes the split sizes and dataset dimensions used by icsor under the same authoritative train-test row split, including the notebook-owned Optuna tuning subset.",
    pd.DataFrame([icsor_split_sizes]),
)
record_setup_table(
    "Benchmark contract checks",
    "This table confirms whether icsor and the classical benchmark share the same row splits, the same feature columns, the same fractional effluent targets, and an Optuna subset restricted to the shared training pool.",
    benchmark_contract_summary,
)

Notebook-managed ML orchestration is ready.
Classical regressors and icsor now share the same operational-plus-fractional input basis and the same fractional effluent target basis for native training.
Measured-output comparison remains available downstream through the external composition-matrix collapse layer.
Shared split indices aligned across icsor and the classical benchmark: True
Shared feature columns aligned across icsor and the classical benchmark: True
Shared target columns aligned across icsor and the classical benchmark: True
Optuna subset drawn only from the shared training pool: True
ICSOR tuning split indices aligned with the notebook-owned Optuna split: True
Saved orchestration tables under timestamp: 20260419_093159
ML orchestration settings
This table shows the shared orchestration hyperparameters and benchmark toggles that control the train-test split, the optional Optuna subset, and whether tuned classical runs are enabled.


c:\Users\eselerio\projects\pibre-model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,random_seed,test_fraction,optuna_dataset_fraction,optuna_test_fraction,tuning_epochs,n_trials,timeout_seconds,use_optuna
0,42,0.2,0.5,0.2,500,100,None,False


Classical benchmark split summary
This table summarizes the split sizes and dataset dimensions used by the classical benchmark models after aligning their native inputs and native targets with icsor.


,train,test,fractional_input_features,fractional_output_targets,fractional_constraints
0,8000,2000,22,20,20


icsor split summary
This table summarizes the split sizes and dataset dimensions used by icsor under the same authoritative train-test row split, including the notebook-owned Optuna tuning subset.


,train,test,optuna_train,optuna_test,fractional_features,fractional_targets,fractional_constraints
0,8000,2000,3200,800,22,20,20


Benchmark contract checks
This table confirms whether icsor and the classical benchmark share the same row splits, the same feature columns, the same fractional effluent targets, and an Optuna subset restricted to the shared training pool.


,classical_feature_space,icsor_feature_space,shared_target_space,direct_comparison_space,shared_split_indices,shared_feature_columns,shared_target_columns,optuna_subset_from_training_pool_only,classical_optuna_enabled,icsor_optuna_split_aligned
0,operational_plus_fractional_influent,operational_plus_fractional_influent,fractional_effluent_components,externally_collapsed_measured_outputs,True,True,True,True,False,True


,classical_feature_space,icsor_feature_space,shared_target_space,direct_comparison_space,shared_split_indices,shared_feature_columns,shared_target_columns,optuna_subset_from_training_pool_only,classical_optuna_enabled,icsor_optuna_split_aligned
0,operational_plus_fractional_influent,operational_plus_fractional_influent,fractional_effluent_components,externally_collapsed_measured_outputs,True,True,True,True,False,True


## icsor

This section follows the strict fractional-space icsor formulation. The notebook passes operational variables and influent ASM1 fractions into the bilinear model, derives the invariant matrix from the Petersen null space, and collapses the projected fractional prediction into the measured composite space with the composition matrix.

ICSOR now supports a manual affine-core estimator choice. OLS keeps the closed-form projected fit, while Ridge applies L2 shrinkage to the identifiable affine core before the same invariant and nonnegative deployment stages. If `USE_OPTUNA` is enabled and the selected estimator is Ridge, the notebook tunes `ridge_alpha` on the notebook-managed Optuna split.

In [5]:
import pandas as pd

from src.models.ml import load_icsor_params, run_icsor_pipeline
from src.utils.analysis import build_notebook_table_recorder, persist_icsor_training_context
from src.utils.simulation import make_simulation_timestamp
from src.utils.train import tune_icsor_hyperparameters

icsor_params = load_icsor_params()
icsor_hyperparameters = dict(icsor_params["training_defaults"])
icsor_estimator = str(icsor_hyperparameters.get("affine_estimator", "ols")).strip().lower()
icsor_hyperparameters["objective"] = "projected_ridge" if icsor_estimator == "ridge" else "projected_ols"
icsor_optuna_enabled = USE_OPTUNA and bool(icsor_params.get("search_space")) and icsor_estimator == "ridge"
icsor_optuna_summary = None

if icsor_optuna_enabled:
    icsor_hyperparameters, icsor_optuna_summary = tune_icsor_hyperparameters(
        icsor_tuning_dataset_splits.train,
        icsor_tuning_dataset_splits.test,
        A_matrix=icsor_A_matrix,
        composition_matrix=composition_matrix,
        measured_output_columns=list(metadata["measured_output_columns"]),
        model_params=icsor_params,
        model_hyperparameters=icsor_hyperparameters,
        n_trials=int(ml_orchestration["n_trials"]),
        timeout=ml_orchestration.get("timeout_seconds"),
        show_progress_bar=True,
    )

icsor_training_timestamp = make_simulation_timestamp()
record_icsor_training_table = build_notebook_table_recorder(
    "training/icsor/summary",
    repo_root=repo_root,
    timestamp=icsor_training_timestamp,
)

icsor_result = run_icsor_pipeline(
    icsor_dataset_splits.train,
    icsor_dataset_splits.test,
    icsor_A_matrix,
    composition_matrix=composition_matrix,
    measured_output_columns=list(metadata["measured_output_columns"]),
    composition_source=metadata.get("composition_source"),
    model_params=icsor_params,
    model_hyperparameters=icsor_hyperparameters,
    optuna_summary=icsor_optuna_summary,
    persist_artifacts=True,
)
icsor_coefficient_inference = dict(icsor_result["coefficient_inference"])
icsor_coefficient_inference_metadata = pd.DataFrame(
    [
        {
            "method": icsor_coefficient_inference["method"],
            "requested_method": icsor_coefficient_inference.get("requested_method"),
            "affine_estimator": icsor_coefficient_inference.get("affine_estimator"),
            "ridge_alpha": icsor_coefficient_inference.get("ridge_alpha"),
            "confidence_level": icsor_coefficient_inference["confidence_level"],
            "coefficient_target": icsor_coefficient_inference["coefficient_target"],
            "rank_deficient": icsor_coefficient_inference["rank_deficient"],
            "design_rank": icsor_coefficient_inference["design_rank"],
            "design_dimension": icsor_coefficient_inference["design_dimension"],
            "degrees_of_freedom": icsor_coefficient_inference["degrees_of_freedom"],
            "note": icsor_coefficient_inference.get("note"),
        }
    ]
)
icsor_training_context_artifacts = persist_icsor_training_context(
    icsor_result,
    {"train": icsor_dataset_splits.train, "test": icsor_dataset_splits.test},
    repo_root=repo_root,
    timestamp=icsor_training_timestamp,
)

print("icsor training complete.")
print(f"Affine-core estimator: {icsor_result['best_hyperparameters'].get('affine_estimator', 'ols')}")
print(f"Operational inputs: {', '.join(metadata['operational_columns'])}")
print(f"Fractional influent states: {', '.join(metadata['state_columns'])}")
print(f"Fractional effluent targets: {', '.join(icsor_dataset.targets.columns)}")
print(f"Measured comparison outputs: {', '.join(metadata['measured_output_columns'])}")
print(f"icsor split sizes: {icsor_split_sizes}")
if icsor_optuna_summary is None:
    print("Optuna tuning not used for this icsor run.")
else:
    print(f"Tuned ridge alpha: {icsor_result['best_hyperparameters'].get('ridge_alpha')}")
    print(f"Saved Optuna summary: {icsor_result['artifact_paths']['optuna']}")
print(f"Saved model bundle: {icsor_result['artifact_paths']['model_bundle']}")
print(f"Saved metrics summary: {icsor_result['artifact_paths']['metrics']}")
print(
    "Saved notebook icsor context bundle under timestamp: "
    f"{icsor_training_context_artifacts['artifact_timestamp']}"
)

record_icsor_training_table(
    "icsor report metadata",
    "This table explains the icsor reporting contract. Fractional-space outputs are native to training and inference, while externally collapsed measured-output metrics are the direct comparison layer.",
    icsor_result["test_report"]["report_metadata"],
)
record_icsor_training_table(
    "icsor hyperparameters",
    "This table lists the hyperparameters used to fit the icsor model in the current run.",
    pd.DataFrame([icsor_result["best_hyperparameters"]]),
)
if icsor_optuna_summary is not None:
    record_icsor_training_table(
        "icsor optuna summary",
        "This table summarizes the ridge-alpha Optuna search used for the current icsor run.",
        pd.DataFrame(
            [
                {
                    "best_value": icsor_optuna_summary["best_value"],
                    "best_trial_number": icsor_optuna_summary["best_trial_number"],
                    "n_trials": icsor_optuna_summary["n_trials"],
                    **{
                        f"best_param_{parameter_name}": value
                        for parameter_name, value in icsor_optuna_summary["best_params"].items()
                    },
                }
            ]
        ),
    )
record_icsor_training_table(
    "icsor coefficient inference metadata",
    "This table records how coefficient uncertainty was estimated for the component-space affine operator and whether the icsor design was rank deficient.",
    icsor_coefficient_inference_metadata,
)
record_icsor_training_table(
    "icsor split summary",
    "This table summarizes the training and testing split sizes and the dimensionality of the icsor dataset used in this run.",
    pd.DataFrame([icsor_split_sizes]),
)
record_icsor_training_table(
    "icsor artifact paths",
    "This table lists the artifact paths produced by the icsor run, including the saved model bundle, metric summary, and Optuna summary when tuning is used.",
    pd.Series(icsor_result["artifact_paths"], name="path").to_frame(),
)
record_icsor_training_table(
    "icsor native component operator",
    "This table shows the identity component-space operator carried by the current icsor bundle before any external measured-output collapse is applied.",
    pd.DataFrame(
        icsor_result["model_bundle"]["collapse_operator"],
        index=metadata["state_columns"],
        columns=metadata["state_columns"],
    ),
)
record_icsor_training_table(
    "icsor invariant carry-through operator",
    "This table shows the conserved-state carry-through operator stored in the icsor bundle. Under the native fractional contract it is expected to remain zero before the external measured-output collapse stage.",
    pd.DataFrame(
        icsor_result["model_bundle"]["pass_through_operator"],
        index=metadata["state_columns"],
        columns=metadata["state_columns"],
    ),
)
record_icsor_training_table(
    "icsor train aggregate measured metrics",
    "This table reports the train-split externally collapsed measured-output metrics for icsor. These aggregate values are directly comparable with the classical benchmark models.",
    icsor_result["train_report"]["aggregate_metrics"],
)
record_icsor_training_table(
    "icsor test aggregate measured metrics",
    "This table reports the test-split externally collapsed measured-output metrics for icsor. These aggregate values are directly comparable with the classical benchmark models.",
    icsor_result["test_report"]["aggregate_metrics"],
)
record_icsor_training_table(
    "icsor test measured per-target metrics",
    "This table breaks down the icsor test metrics by the externally collapsed measured-output target labels.",
    icsor_result["test_report"]["per_target_metrics"],
)
record_icsor_training_table(
    "icsor train aggregate fractional metrics",
    "This table reports the train-split native fractional-space metrics for icsor.",
    icsor_result["train_report"]["fractional_aggregate_metrics"],
)
record_icsor_training_table(
    "icsor test aggregate fractional metrics",
    "This table reports the test-split native fractional-space metrics for icsor.",
    icsor_result["test_report"]["fractional_aggregate_metrics"],
)
record_icsor_training_table(
    "icsor test fractional per-target metrics",
    "This table breaks down the icsor test metrics by ASM fractional effluent target.",
    icsor_result["test_report"]["fractional_per_target_metrics"],
)
record_icsor_training_table(
    "icsor test prediction uncertainty summary",
    "This table summarizes the average widths and standard errors of the icsor affine-core component-space confidence and prediction intervals on the test split. These are not exact intervals for the final nonnegativity-corrected predictor when the correction activates.",
    icsor_result["test_report"]["prediction_uncertainty_summary"],
)
record_icsor_training_table(
    "icsor test diagnostic summary",
    "This table summarizes icsor-native diagnostics, including fractional constraint residuals and raw-to-projected adjustment magnitudes. These diagnostics complement the direct comparison metrics but are not themselves apples-to-apples rankings against the classical models.",
    icsor_result["test_report"]["diagnostic_summary"],
)
record_icsor_training_table(
    "icsor test constraint residual summary",
    "This table summarizes the sample-level fractional constraint residual norms on the icsor test split.",
    icsor_result["test_report"]["constraint_residuals"].describe().T,
)

icsor training complete.
Affine-core estimator: ols
Operational inputs: HRT, Aeration
Fractional influent states: S_O, S_F, S_A, S_NH4, S_NO2, S_NO3, S_N2, S_PO4, S_I, S_ALK, X_I, X_S, X_H, X_PAO, X_PP, X_PHA, X_AOB, X_NOB, X_MeP, X_MeOH
Fractional effluent targets: Out_S_O, Out_S_F, Out_S_A, Out_S_NH4, Out_S_NO2, Out_S_NO3, Out_S_N2, Out_S_PO4, Out_S_I, Out_S_ALK, Out_X_I, Out_X_S, Out_X_H, Out_X_PAO, Out_X_PP, Out_X_PHA, Out_X_AOB, Out_X_NOB, Out_X_MeP, Out_X_MeOH
Measured comparison outputs: COD, TN, TP, TSS
icsor split sizes: {'train': 8000, 'test': 2000, 'optuna_train': 3200, 'optuna_test': 800, 'fractional_features': 22, 'fractional_targets': 20, 'fractional_constraints': 20}
Optuna tuning not used for this icsor run.
Saved model bundle: C:\Users\eselerio\projects\pibre-model\results\icsor\model_20260419_093258.pkl
Saved metrics summary: C:\Users\eselerio\projects\pibre-model\results\icsor\metrics_20260419_093258.json
Saved notebook icsor context bundle under timestamp: 20260419_

,native_prediction_space,comparison_target_space,constraint_space,direct_comparison_scope,diagnostic_scope,projection_active,constraint_status
0,fractional,external_measured_output,fractional,externally_collapsed_measured_output_metrics,model_native_fractional_space_nonnegative_lp_d...,True,active_nonnegative_lp


icsor hyperparameters
This table lists the hyperparameters used to fit the icsor model in the current run.


,objective,solver,affine_estimator,ols_backend,ridge_alpha,include_bias_term,lstsq_rcond,projection_solver,constraint_tolerance,nonnegativity_tolerance,measured_deviation_weight,component_deviation_weight,tradeoff_parameter,highs_presolve,highs_max_iter,highs_verbose,highs_retry_without_presolve,uncertainty_method,confidence_level
0,projected_ols,multivariate_lstsq,ols,numpy_lstsq,0.01,True,None,highs,1.000000e-08,1.000000e-10,1.0,1.0,1.000000e-09,True,10000,False,True,analytic,0.95


icsor coefficient inference metadata
This table records how coefficient uncertainty was estimated for the component-space affine operator and whether the icsor design was rank deficient.


...

icsor test diagnostic summary
This table summarizes icsor-native diagnostics, including fractional constraint residuals and raw-to-projected adjustment magnitudes. These diagnostics complement the direct comparison metrics but are not themselves apples-to-apples rankings against the classical models.


,diagnostic_name,prediction_type,constraint_mean_l2,constraint_max_l2,constraint_mean_abs,constraint_max_abs,mean_l2,max_l2,mean_abs,max_abs
0,fractional_constraint_residual,raw,6.285251e-01,3.961815e+00,1.939229e-01,3.760582e+00,NaN,NaN,NaN,NaN
1,fractional_constraint_residual,affine,1.662987e-12,7.513713e-12,4.749364e-13,7.275958e-12,NaN,NaN,NaN,NaN
2,fractional_constraint_residual,projected,1.693526e-12,2.100379e-11,4.841045e-13,1.989520e-11,NaN,NaN,NaN,NaN
3,measured_raw_to_affine_adjustment,raw_to_projected,NaN,NaN,NaN,NaN,0.050348,0.256464,0.019904,0.191587
4,measured_affine_to_projected_adjustment,raw_to_projected,NaN,NaN,NaN,NaN,3.131916,32.932409,0.908837,32.799793
5,measured_raw_to_projected_adjustment,raw_to_projected,NaN,NaN,NaN,NaN,3.146415,32.931779,0.924304,32.795325
6,fractional_raw_to_affine_adjustment,raw_to_projected,NaN,NaN,NaN,NaN,0.026813,0.128706,0.003499,0.063912
7,fractional_affine_to_projected_adjustment,raw_to_projected,NaN,NaN,NaN,NaN,2.845795,24.239938,0.229453,17.012664
8,fractional_raw_to_projected_adjustment,raw_to_projected,NaN,NaN,NaN,NaN,2.851225,24.239963,0.232230,17.019054


icsor test constraint residual summary
This table summarizes the sample-level fractional constraint residual norms on the icsor test split.


,count,mean,std,min,25%,50%,75%,max
raw_constraint_l2,2000.0,6.285251e-01,4.231273e-01,0.047919,3.508808e-01,5.406359e-01,7.851704e-01,3.961815e+00
affine_constraint_l2,2000.0,1.662987e-12,1.035270e-12,0.000000,9.647498e-13,1.401743e-12,2.068586e-12,7.513713e-12
projected_constraint_l2,2000.0,1.693526e-12,1.136531e-12,0.000000,9.647695e-13,1.460343e-12,2.087013e-12,2.100379e-11


,count,mean,std,min,25%,50%,75%,max
raw_constraint_l2,2000.0,6.285251e-01,4.231273e-01,0.047919,3.508808e-01,5.406359e-01,7.851704e-01,3.961815e+00
affine_constraint_l2,2000.0,1.662987e-12,1.035270e-12,0.000000,9.647498e-13,1.401743e-12,2.068586e-12,7.513713e-12
projected_constraint_l2,2000.0,1.693526e-12,1.136531e-12,0.000000,9.647695e-13,1.460343e-12,2.087013e-12,2.100379e-11


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display as ipython_display

from src.utils.analysis import load_latest_icsor_training_context
from src.utils.plot import persist_figure_artifacts, plot_train_test_parity_panels
from src.utils.process import collapse_fractional_states_to_measured_outputs
from src.utils.simulation import make_simulation_timestamp

if "icsor_result" in globals() and "icsor_dataset_splits" in globals():
    icsor_notebook_context = {
        "train_report": icsor_result["train_report"],
        "test_report": icsor_result["test_report"],
        "train_targets": icsor_dataset_splits.train.targets,
        "test_targets": icsor_dataset_splits.test.targets,
        "train_constraint_reference": icsor_dataset_splits.train.constraint_reference,
        "test_constraint_reference": icsor_dataset_splits.test.constraint_reference,
    }
    print("Using in-memory icsor training outputs for the parity plots.")
else:
    loaded_icsor_context = load_latest_icsor_training_context(repo_root=repo_root)
    icsor_notebook_context = {
        "artifact_timestamp": loaded_icsor_context["artifact_timestamp"],
        "train_report": loaded_icsor_context["train_report"],
        "test_report": loaded_icsor_context["test_report"],
        "train_targets": loaded_icsor_context["train_targets"],
        "test_targets": loaded_icsor_context["test_targets"],
        "train_constraint_reference": loaded_icsor_context["train_constraint_reference"],
        "test_constraint_reference": loaded_icsor_context["test_constraint_reference"],
    }
    print(
        "icsor parity fallback: loaded the latest saved training context from "
        f"{loaded_icsor_context['artifact_timestamp']}."
    )

icsor_parity_timestamp = make_simulation_timestamp()
icsor_actual_composite_train = collapse_fractional_states_to_measured_outputs(
    icsor_notebook_context["train_targets"].to_numpy(dtype=float),
    list(metadata["state_columns"]),
    composition_matrix,
    list(metadata["measured_output_columns"]),
    output_prefix="Out_",
    index=icsor_notebook_context["train_targets"].index,
)
icsor_actual_composite_test = collapse_fractional_states_to_measured_outputs(
    icsor_notebook_context["test_targets"].to_numpy(dtype=float),
    list(metadata["state_columns"]),
    composition_matrix,
    list(metadata["measured_output_columns"]),
    output_prefix="Out_",
    index=icsor_notebook_context["test_targets"].index,
)
icsor_projected_composite_train = (
    icsor_notebook_context["train_report"]["projected_predictions"]
    .rename(columns=lambda column_name: str(column_name).removeprefix("Projected_"))
    .loc[:, icsor_actual_composite_train.columns]
)
icsor_projected_composite_test = (
    icsor_notebook_context["test_report"]["projected_predictions"]
    .rename(columns=lambda column_name: str(column_name).removeprefix("Projected_"))
    .loc[:, icsor_actual_composite_test.columns]
)
icsor_projected_fractional_train = (
    icsor_notebook_context["train_report"]["projected_fractional_predictions"]
    .rename(columns=lambda column_name: str(column_name).removeprefix("ProjectedFractional_"))
    .loc[:, icsor_notebook_context["train_constraint_reference"].columns]
)
icsor_projected_fractional_test = (
    icsor_notebook_context["test_report"]["projected_fractional_predictions"]
    .rename(columns=lambda column_name: str(column_name).removeprefix("ProjectedFractional_"))
    .loc[:, icsor_notebook_context["test_constraint_reference"].columns]
)

print("The measured-output parity figure is the direct benchmark-comparison view for icsor.")
print("The ASM fractional parity figure is the icsor-native feasibility view after projection.")

figure, _ = plot_train_test_parity_panels(
    icsor_actual_composite_train,
    icsor_projected_composite_train,
    icsor_actual_composite_test,
    icsor_projected_composite_test,
    title="icsor projected train-test parity for measured composite outputs",
    x_label="Actual measured value",
    y_label="Projected measured prediction",
    figure_size_per_panel=(4.2, 3.6),
)
persist_figure_artifacts(
    figure,
    "training/icsor/parity",
    "measured_output_train_test_parity",
    repo_root=repo_root,
    timestamp=icsor_parity_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_train_test_parity_panels(
    icsor_notebook_context["train_constraint_reference"],
    icsor_projected_fractional_train,
    icsor_notebook_context["test_constraint_reference"],
    icsor_projected_fractional_test,
    title="icsor projected train-test parity for ASM fractional components",
    x_label="Actual ASM fractional value",
    y_label="Projected ASM fractional prediction",
    figure_size_per_panel=(4.2, 3.6),
    max_columns=5,
)
persist_figure_artifacts(
    figure,
    "training/icsor/parity",
    "asm_fractional_train_test_parity",
    repo_root=repo_root,
    timestamp=icsor_parity_timestamp,
)
ipython_display(figure)
plt.close(figure)

In [ ]:
from src.utils.analysis import (
    build_notebook_table_recorder,
    build_separated_negative_prediction_tables,
    load_latest_icsor_training_context,
)
from src.utils.simulation import make_simulation_timestamp

if "icsor_result" in globals():
    icsor_negative_report_bundle = {
        "train": icsor_result["train_report"],
        "test": icsor_result["test_report"],
    }
    print("Using in-memory icsor reports for the negative-prediction summaries.")
else:
    loaded_icsor_context = load_latest_icsor_training_context(repo_root=repo_root)
    icsor_negative_report_bundle = {
        "train": loaded_icsor_context["train_report"],
        "test": loaded_icsor_context["test_report"],
    }
    print(
        "icsor negative-prediction fallback: loaded the latest saved training context from "
        f"{loaded_icsor_context['artifact_timestamp']}."
    )

icsor_negative_timestamp = make_simulation_timestamp()
record_icsor_negative_table = build_notebook_table_recorder(
    "training/icsor/negative_predictions",
    repo_root=repo_root,
    timestamp=icsor_negative_timestamp,
    index=False,
)
icsor_negative_prediction_tables = build_separated_negative_prediction_tables(
    icsor_negative_report_bundle
)

projected_test_composite_summary = icsor_negative_prediction_tables["composite"]["projected"]["summary"].loc[
    lambda frame: frame["split"] == "test"
].iloc[0]
projected_test_fractional_summary = icsor_negative_prediction_tables["fractional"]["projected"]["summary"].loc[
    lambda frame: frame["split"] == "test"
].iloc[0]


def print_negative_prediction_headline(summary_row, *, label):
    print(
        f"Projected icsor test {label} predictions are negative in "
        f"{int(summary_row['negative_predictions'])} of "
        f"{int(summary_row['total_predictions'])} predicted values "
        f"({summary_row['negative_prediction_rate_pct']:.2f}%)."
    )
    print(
        "Those negatives occur in "
        f"{int(summary_row['samples_with_any_negative'])} of "
        f"{int(summary_row['total_samples'])} test samples "
        f"({summary_row['sample_incidence_rate_pct']:.2f}%)."
    )
    print(
        f"The most negative projected test {label} prediction is "
        f"{summary_row['minimum_prediction']:.6f}."
    )


print_negative_prediction_headline(projected_test_composite_summary, label="composite")
print()
print_negative_prediction_headline(projected_test_fractional_summary, label="ASM fractional")

family_display_names = {
    "composite": "composite",
    "fractional": "ASM fractional",
}
family_space_descriptions = {
    "composite": "composite measured-output space",
    "fractional": "ASM fractional component space",
}
family_item_labels = {
    "composite": "target",
    "fractional": "component",
}

for family_name in ["composite", "fractional"]:
    family_tables = icsor_negative_prediction_tables[family_name]
    family_display_name = family_display_names[family_name]
    family_space_description = family_space_descriptions[family_name]
    item_label = family_item_labels[family_name]

    for prediction_type in ["raw", "affine", "projected"]:
        prediction_tables = family_tables[prediction_type]
        summary_table = prediction_tables["summary"]
        per_target_table = prediction_tables["per_target"]
        if item_label == "component":
            per_target_table = per_target_table.rename(columns={"target": "component"})

        record_icsor_negative_table(
            f"icsor {family_display_name} {prediction_type} negative prediction incidence summary",
            f"This table summarizes how often icsor {prediction_type} predictions in the {family_space_description} are negative on the train and test splits.",
            summary_table,
        )
        record_icsor_negative_table(
            f"icsor {family_display_name} {prediction_type} negative prediction incidence by {item_label}",
            f"This table breaks down the icsor {prediction_type} negative-prediction incidence across each {item_label} in the {family_space_description}, including counts, rates, and observed negative magnitudes.",
            per_target_table,
        )

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display as ipython_display

from src.utils.analysis import load_latest_icsor_training_context
from src.utils.plot import (
    persist_figure_artifacts,
    plot_coefficient_bar_chart,
    plot_coefficient_heatmap,
    plot_coefficient_tensor_heatmaps,
)
from src.utils.simulation import make_simulation_timestamp

if "icsor_result" in globals():
    component_coefficients = icsor_result["model_bundle"]["component_coefficients"]
    print("Using in-memory icsor coefficient outputs.")
else:
    loaded_icsor_context = load_latest_icsor_training_context(repo_root=repo_root)
    component_coefficients = loaded_icsor_context["component_coefficients"]
    print(
        "icsor coefficient fallback: loaded the latest saved training context from "
        f"{loaded_icsor_context['artifact_timestamp']}."
    )

icsor_coefficient_timestamp = make_simulation_timestamp()
component_targets = list(metadata["state_columns"])
operational_variables = list(metadata["operational_columns"])
influent_fractional_variables = list(metadata["state_columns"])

figure, _ = plot_coefficient_heatmap(
    component_coefficients["W_u"],
    row_labels=component_targets,
    column_labels=operational_variables,
    title=r"icsor component-space operational coefficients ($W_u$)",
    x_label="Operational variable",
    y_label="ASM fractional target",
)
persist_figure_artifacts(
    figure,
    "training/icsor/coefficients",
    "w_u_heatmap",
    repo_root=repo_root,
    timestamp=icsor_coefficient_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_coefficient_heatmap(
    component_coefficients["W_in"],
    row_labels=component_targets,
    column_labels=influent_fractional_variables,
    title=r"icsor component-space influent coefficients ($W_{in}$)",
    x_label="Influent fractional variable",
    y_label="ASM fractional target",
)
persist_figure_artifacts(
    figure,
    "training/icsor/coefficients",
    "w_in_heatmap",
    repo_root=repo_root,
    timestamp=icsor_coefficient_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_coefficient_bar_chart(
    component_coefficients["b"],
    labels=component_targets,
    title=r"icsor component-space bias coefficients ($b$)",
    x_label="ASM fractional target",
    y_label="Coefficient value",
)
persist_figure_artifacts(
    figure,
    "training/icsor/coefficients",
    "bias_bar_chart",
    repo_root=repo_root,
    timestamp=icsor_coefficient_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_coefficient_tensor_heatmaps(
    component_coefficients["Theta_uu"],
    target_labels=component_targets,
    row_labels=operational_variables,
    column_labels=operational_variables,
    title=r"icsor component-space operational interaction coefficients ($\Theta_{uu}$)",
    x_label="Operational variable",
    y_label="Operational variable",
)
persist_figure_artifacts(
    figure,
    "training/icsor/coefficients",
    "theta_uu_heatmaps",
    repo_root=repo_root,
    timestamp=icsor_coefficient_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_coefficient_tensor_heatmaps(
    component_coefficients["Theta_cc"],
    target_labels=component_targets,
    row_labels=influent_fractional_variables,
    column_labels=influent_fractional_variables,
    title=r"icsor component-space influent interaction coefficients ($\Theta_{cc}$)",
    x_label="Influent fractional variable",
    y_label="Influent fractional variable",
)
persist_figure_artifacts(
    figure,
    "training/icsor/coefficients",
    "theta_cc_heatmaps",
    repo_root=repo_root,
    timestamp=icsor_coefficient_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_coefficient_tensor_heatmaps(
    component_coefficients["Theta_uc"],
    target_labels=component_targets,
    row_labels=operational_variables,
    column_labels=influent_fractional_variables,
    title=r"icsor component-space operational-influent interaction coefficients ($\Theta_{uc}$)",
    x_label="Influent fractional variable",
    y_label="Operational variable",
)
persist_figure_artifacts(
    figure,
    "training/icsor/coefficients",
    "theta_uc_heatmaps",
    repo_root=repo_root,
    timestamp=icsor_coefficient_timestamp,
)
ipython_display(figure)
plt.close(figure)

In [ ]:
import pickle

import numpy as np
import pandas as pd

from src.utils.analysis import build_notebook_table_recorder, load_latest_icsor_training_context
from src.utils.simulation import make_simulation_timestamp

coefficient_density_block_specs = [
    ("b", r"$b$", "Bias block"),
    ("W_u", r"$W_u$", "Operational first-order block"),
    ("W_in", r"$W_{in}$", "Influent first-order block"),
    ("Theta_uu", r"$\Theta_{uu}$", "Operational-operational interaction block"),
    ("Theta_uc", r"$\Theta_{uc}$", "Operational-influent interaction block"),
    ("Theta_cc", r"$\Theta_{cc}$", "Influent-influent interaction block"),
]
coefficient_density_retention_fraction = 0.0001
coefficient_density_threshold_rule = (
    f"A coefficient is retained when its absolute value is at least {coefficient_density_retention_fraction:g} times "
    "the maximum absolute identifiable coefficient within the same target equation "
    f"({100.0 * coefficient_density_retention_fraction:.1f}% of the target-equation maximum)."
)

if "icsor_result" in globals():
    identifiable_coefficients = icsor_result["model_bundle"]["identifiable_coefficients"]
    print("Using in-memory icsor model bundle for the icsor coefficient density analysis.")
else:
    loaded_icsor_context = load_latest_icsor_training_context(repo_root=repo_root)
    icsor_model_path = loaded_icsor_context["training_artifact_paths"].get("model_bundle")
    if icsor_model_path is None:
        raise ValueError("icsor model bundle path is required for the icsor coefficient density analysis.")
    with icsor_model_path.open("rb") as model_file:
        loaded_model_bundle = pickle.load(model_file)
    identifiable_coefficients = loaded_model_bundle["identifiable_coefficients"]
    print(
        "icsor coefficient-density fallback: loaded the latest saved model bundle from "
        f"{icsor_model_path}."
    )

coefficient_density_block_arrays = {
    block_key: np.asarray(identifiable_coefficients[block_key], dtype=float)
    for block_key, _, _ in coefficient_density_block_specs
}
target_equation_count = int(coefficient_density_block_arrays["b"].shape[0])
target_equation_labels = (
    list(metadata.get("state_columns", []))
    if "metadata" in globals()
    else [f"target_{target_index + 1}" for target_index in range(target_equation_count)]
)
target_equation_max_abs_values = []
for target_index in range(target_equation_count):
    target_equation_abs_values = []
    for block_key, _, _ in coefficient_density_block_specs:
        block_array = coefficient_density_block_arrays[block_key]
        target_slice = (
            block_array[target_index : target_index + 1]
            if block_array.ndim == 1
            else block_array[target_index]
        )
        target_equation_abs_values.append(np.abs(np.asarray(target_slice, dtype=float).ravel()))
    target_equation_max_abs_values.append(
        max(float(target_values.max()) if target_values.size else 0.0 for target_values in target_equation_abs_values)
    )

target_equation_max_abs_values = np.asarray(target_equation_max_abs_values, dtype=float)
target_equation_thresholds = coefficient_density_retention_fraction * target_equation_max_abs_values
icsor_target_equation_threshold_summary = pd.DataFrame(
    {
        "target": target_equation_labels,
        "target_equation_max_abs_coefficient": target_equation_max_abs_values,
        "retention_threshold": target_equation_thresholds,
    }
)

coefficient_density_rows = []
for block_key, block_label, block_description in coefficient_density_block_specs:
    block_array = coefficient_density_block_arrays[block_key]
    selectable_coefficient_count = 0
    retained_coefficient_count = 0
    for target_index in range(target_equation_count):
        target_slice = (
            block_array[target_index : target_index + 1]
            if block_array.ndim == 1
            else block_array[target_index]
        )
        absolute_target_values = np.abs(np.asarray(target_slice, dtype=float).ravel())
        selectable_coefficient_count += int(absolute_target_values.size)
        if target_equation_thresholds[target_index] != 0.0:
            retained_coefficient_count += int(np.sum(absolute_target_values >= target_equation_thresholds[target_index]))
    coefficient_density_rows.append(
        {
            "block_key": block_key,
            "block_label": block_label,
            "block_description": block_description,
            "selectable_coefficients": selectable_coefficient_count,
            "retained_coefficients": retained_coefficient_count,
            "retention_pct": 100.0 * retained_coefficient_count / selectable_coefficient_count,
        }
    )

icsor_coefficient_density_table = pd.DataFrame(coefficient_density_rows)
icsor_total_selectable_coefficients = int(icsor_coefficient_density_table["selectable_coefficients"].sum())
icsor_total_retained_coefficients = int(icsor_coefficient_density_table["retained_coefficients"].sum())
icsor_total_retention_pct = 100.0 * icsor_total_retained_coefficients / icsor_total_selectable_coefficients
icsor_coefficient_density_table = pd.concat(
    [
        icsor_coefficient_density_table,
        pd.DataFrame(
            [
                {
                    "block_key": "total",
                    "block_label": "Total",
                    "block_description": "All identifiable affine-core blocks",
                    "selectable_coefficients": icsor_total_selectable_coefficients,
                    "retained_coefficients": icsor_total_retained_coefficients,
                    "retention_pct": icsor_total_retention_pct,
                }
            ]
        ),
    ],
    ignore_index=True,
)

icsor_coefficient_density_timestamp = make_simulation_timestamp()
record_icsor_coefficient_density_table = build_notebook_table_recorder(
    "training/icsor/coefficient_density",
    repo_root=repo_root,
    timestamp=icsor_coefficient_density_timestamp,
    index=False,
)

equation_threshold_min = float(target_equation_thresholds.min()) if target_equation_thresholds.size else float("nan")
equation_threshold_max = float(target_equation_thresholds.max()) if target_equation_thresholds.size else float("nan")

print("icsor coefficient density analysis complete.")
print(coefficient_density_threshold_rule)
print(
    f"Absolute thresholds across target equations: {equation_threshold_min:.6f} to "
    f"{equation_threshold_max:.6f}."
)
print(
    f"Overall retained coefficients: {icsor_total_retained_coefficients} of "
    f"{icsor_total_selectable_coefficients} ({icsor_total_retention_pct:.2f}%)."
)
print(f"Saved coefficient-density table under timestamp: {icsor_coefficient_density_timestamp}")

record_icsor_coefficient_density_table(
    "icsor coefficient density analysis",
    "This table reports the per-block density of the identifiable component-space icsor affine core. "
    f"{coefficient_density_threshold_rule} The implied absolute thresholds span "
    f"{equation_threshold_min:.6f} to {equation_threshold_max:.6f} across the target equations.",
    icsor_coefficient_density_table.round(
        {
            "retention_pct": 2,
        }
    ),
)
record_icsor_coefficient_density_table(
    "icsor target-equation threshold summary",
    "This table lists the maximum absolute identifiable coefficient in each component-space target equation and the corresponding equation-adaptive retention threshold used in the density analysis.",
    icsor_target_equation_threshold_summary.round(
        {
            "target_equation_max_abs_coefficient": 6,
            "retention_threshold": 6,
        }
    ),
)


## icsor Response Surface

This block fixes the influent fractional states to a common profile and then evaluates the trained icsor model over an extended HRT-Aeration grid. By default the fixed influent profile uses the midpoint of each configured influent-state range, and the operational grid extends 50% beyond the original training domain while clipping the lower HRT and Aeration bounds at zero so the operating points remain physically valid.

Override any of the defaults by defining `icsor_response_surface_overrides = {...}` before running the next cell.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display as ipython_display

from src.utils.analysis import (
    build_icsor_response_surface_prediction_data,
    build_notebook_table_recorder,
    load_icsor_response_surface_defaults,
    load_latest_icsor_training_context,
)
from src.utils.plot import persist_figure_artifacts, plot_response_surface_contours
from src.utils.simulation import make_simulation_timestamp

icsor_response_surface_defaults = load_icsor_response_surface_defaults()
icsor_response_surface_overrides = dict(globals().get("icsor_response_surface_overrides", {}))
icsor_response_surface_config = {
    **icsor_response_surface_defaults,
    **icsor_response_surface_overrides,
    }
response_surface_decimal_places = 2
icsor_response_surface_timestamp = make_simulation_timestamp()
record_icsor_response_surface_table = build_notebook_table_recorder(
    "training/icsor/response_surface",
    repo_root=repo_root,
    timestamp=icsor_response_surface_timestamp,
    index=False,
)

if "icsor_result" in globals():
    icsor_model_path = icsor_result["artifact_paths"]["model_bundle"]
    print("Using in-memory icsor model bundle for the response-surface analysis.")
else:
    loaded_icsor_context = load_latest_icsor_training_context(repo_root=repo_root)
    icsor_model_path = loaded_icsor_context["training_artifact_paths"].get("model_bundle")
    print(
        "icsor response-surface fallback: loaded the latest saved training context from "
        f"{loaded_icsor_context['artifact_timestamp']}."
    )

if icsor_model_path is None:
    raise ValueError("icsor model bundle path is required to generate the response-surface plot.")

icsor_response_surface_result = build_icsor_response_surface_prediction_data(
    icsor_model_path,
    metadata=metadata,
    grid_points_per_axis=int(icsor_response_surface_config["grid_points_per_axis"]),
    operational_extension_fraction=float(
        icsor_response_surface_config["operational_extension_fraction"]
        ),
    fixed_influent_profile=icsor_response_surface_config["fixed_influent_profile"],
    )

projected_composite_columns = [
    "HRT",
    "Aeration",
    *[
        f"Projected_Out_{target_name}"
        for target_name in metadata["measured_output_columns"]
        ],
]
projected_composite_preview = (
    icsor_response_surface_result["prediction_table"]
    .loc[:, projected_composite_columns]
    .rename(columns=lambda column_name: str(column_name).removeprefix("Projected_Out_"))
    .round(response_surface_decimal_places)
)
projected_composite_surfaces = dict(icsor_response_surface_result["per_target_surfaces"])

print("icsor response-surface analysis complete.")
print(
    "Fixed influent profile strategy: "
    f"{icsor_response_surface_result['response_surface_config']['fixed_influent_profile']}"
    )
print(
    "Extended HRT range: "
    f"{icsor_response_surface_result['extended_domain']['HRT']['min']:.2f} to "
    f"{icsor_response_surface_result['extended_domain']['HRT']['max']:.2f}"
    )
print(
    "Extended Aeration range: "
    f"{icsor_response_surface_result['extended_domain']['Aeration']['min']:.2f} to "
    f"{icsor_response_surface_result['extended_domain']['Aeration']['max']:.2f}"
    )

record_icsor_response_surface_table(
    "icsor response-surface configuration",
    "This table lists the configuration used to build the icsor response-surface grid, including the grid density, the extrapolation range, and the fixed influent-profile strategy.",
    pd.DataFrame([icsor_response_surface_config]).round(response_surface_decimal_places),
    )
record_icsor_response_surface_table(
    "icsor fixed influent profile",
    "This table shows the influent fractional-state profile that is held constant while the HRT-Aeration response surface is evaluated.",
    icsor_response_surface_result["fixed_influent_profile"].to_frame().round(response_surface_decimal_places),
    )
record_icsor_response_surface_table(
    "icsor projected composite concentration preview",
    "This table previews only the projected composite concentrations over the HRT-Aeration response-surface grid.",
    projected_composite_preview.head(),
    )
if "prediction_uncertainty_summary" in icsor_response_surface_result:
    record_icsor_response_surface_table(
        "icsor response-surface prediction uncertainty summary",
        "This table summarizes the average widths and standard errors of the projected icsor response-surface confidence and prediction intervals over the evaluated grid.",
        icsor_response_surface_result["prediction_uncertainty_summary"].round(response_surface_decimal_places),
        )

figure, _ = plot_response_surface_contours(
    icsor_response_surface_result["operational_meshes"]["HRT"],
    icsor_response_surface_result["operational_meshes"]["Aeration"],
    projected_composite_surfaces,
    title="icsor projected composite concentrations across HRT and Aeration",
    x_label="HRT",
    y_label="Aeration",
    training_domain=icsor_response_surface_result["training_domain"],
    contour_levels=int(icsor_response_surface_config["contour_levels"]),
    )
persist_figure_artifacts(
    figure,
    "training/icsor/response_surface",
    "projected_composite_response_surface",
    repo_root=repo_root,
    timestamp=icsor_response_surface_timestamp,
)
ipython_display(figure)
plt.close(figure)

## Additional Classical Regressors

The notebook now benchmarks the classical regressors, including K-nearest neighbors, partial least squares regression, and three fixed-shape artificial neural network variants, on the same train-test rows, the same operational inputs, the same influent fractional-state inputs, and the same fractional effluent targets used by icsor. The direct cross-model comparison remains the externally collapsed measured-output metric layer, while the model-native fractional reports remain available alongside it.

In [ ]:
import pandas as pd

from src.models.ml import (
    load_adaboost_regressor_params,
    load_ann_deep_regressor_params,
    load_ann_medium_regressor_params,
    load_ann_shallow_regressor_params,
    load_catboost_regressor_params,
    load_knn_regressor_params,
    load_lightgbm_regressor_params,
    load_pls_regressor_params,
    load_random_forest_regressor_params,
    load_svr_regressor_params,
    load_xgboost_regressor_params,
    run_adaboost_regressor_pipeline,
    run_ann_deep_regressor_pipeline,
    run_ann_medium_regressor_pipeline,
    run_ann_shallow_regressor_pipeline,
    run_catboost_regressor_pipeline,
    run_knn_regressor_pipeline,
    run_lightgbm_regressor_pipeline,
    run_pls_regressor_pipeline,
    run_random_forest_regressor_pipeline,
    run_svr_regressor_pipeline,
    run_xgboost_regressor_pipeline,
)
from src.models.ml.adaboost_regressor import build_adaboost_regressor_model
from src.models.ml.ann_deep_regressor import build_ann_deep_regressor_model
from src.models.ml.ann_medium_regressor import build_ann_medium_regressor_model
from src.models.ml.ann_shallow_regressor import build_ann_shallow_regressor_model
from src.models.ml.catboost_regressor import build_catboost_regressor_model
from src.models.ml.knn_regressor import build_knn_regressor_model
from src.models.ml.lightgbm_regressor import build_lightgbm_regressor_model
from src.models.ml.pls_regressor import build_pls_regressor_model
from src.models.ml.random_forest_regressor import build_random_forest_regressor_model
from src.models.ml.svr_regressor import build_svr_regressor_model
from src.models.ml.xgboost_regressor import build_xgboost_regressor_model
from src.utils.analysis import (
    build_negative_prediction_tables,
    build_notebook_table_recorder,
    persist_classical_training_context,
)
from src.utils.simulation import make_simulation_timestamp
from src.utils.train import tune_tabular_regressor_hyperparameters

classical_regressor_specs = {
    "xgboost_regressor": {
        "load_params": load_xgboost_regressor_params,
        "build_model": build_xgboost_regressor_model,
        "runner": run_xgboost_regressor_pipeline,
    },
    "lightgbm_regressor": {
        "load_params": load_lightgbm_regressor_params,
        "build_model": build_lightgbm_regressor_model,
        "runner": run_lightgbm_regressor_pipeline,
    },
    "catboost_regressor": {
        "load_params": load_catboost_regressor_params,
        "build_model": build_catboost_regressor_model,
        "runner": run_catboost_regressor_pipeline,
    },
    "adaboost_regressor": {
        "load_params": load_adaboost_regressor_params,
        "build_model": build_adaboost_regressor_model,
        "runner": run_adaboost_regressor_pipeline,
    },
    "random_forest_regressor": {
        "load_params": load_random_forest_regressor_params,
        "build_model": build_random_forest_regressor_model,
        "runner": run_random_forest_regressor_pipeline,
    },
    "svr_regressor": {
        "load_params": load_svr_regressor_params,
        "build_model": build_svr_regressor_model,
        "runner": run_svr_regressor_pipeline,
    },
    "knn_regressor": {
        "load_params": load_knn_regressor_params,
        "build_model": build_knn_regressor_model,
        "runner": run_knn_regressor_pipeline,
    },
    "pls_regressor": {
        "load_params": load_pls_regressor_params,
        "build_model": build_pls_regressor_model,
        "runner": run_pls_regressor_pipeline,
    },
    "ann_shallow_regressor": {
        "load_params": load_ann_shallow_regressor_params,
        "build_model": build_ann_shallow_regressor_model,
        "runner": run_ann_shallow_regressor_pipeline,
    },
    "ann_medium_regressor": {
        "load_params": load_ann_medium_regressor_params,
        "build_model": build_ann_medium_regressor_model,
        "runner": run_ann_medium_regressor_pipeline,
    },
    "ann_deep_regressor": {
        "load_params": load_ann_deep_regressor_params,
        "build_model": build_ann_deep_regressor_model,
        "runner": run_ann_deep_regressor_pipeline,
    },
}

def run_classical_regressor(
    model_name: str,
    *,
    use_optuna: bool = False,
    persist_artifacts: bool = True,
    spec_map: dict | None = None,
):
    resolved_spec_map = classical_regressor_specs if spec_map is None else spec_map
    spec = resolved_spec_map[model_name]
    model_params = spec["load_params"]()
    selected_hyperparameters = dict(model_params["training_defaults"])
    optuna_summary = None

    if use_optuna:
        selected_hyperparameters, optuna_summary = tune_tabular_regressor_hyperparameters(
            model_name,
            spec["build_model"],
            tuning_dataset_splits.train,
            tuning_dataset_splits.test,
            A_matrix=A_matrix,
            composition_matrix=composition_matrix,
            measured_output_columns=list(metadata["measured_output_columns"]),
            model_params=model_params,
            n_trials=int(ml_orchestration["n_trials"]),
            timeout=ml_orchestration["timeout_seconds"],
        )

    result = spec["runner"](
        main_dataset_splits.train,
        main_dataset_splits.test,
        A_matrix,
        composition_matrix=composition_matrix,
        measured_output_columns=list(metadata["measured_output_columns"]),
        model_params=model_params,
        model_hyperparameters=selected_hyperparameters,
        optuna_summary=optuna_summary,
        persist_artifacts=persist_artifacts,
    )

    training_timestamp = make_simulation_timestamp()
    record_training_table = build_notebook_table_recorder(
        f"training/{model_name}/summary",
        repo_root=repo_root,
        timestamp=training_timestamp,
        index=False,
    )
    projection_active = bool(result["test_report"]["report_metadata"].loc[0, "projection_active"])
    negative_prediction_tables = build_negative_prediction_tables(
        {
            "train": result["train_report"],
            "test": result["test_report"],
        }
    )
    result["negative_prediction_summary"] = negative_prediction_tables["summary"]
    result["negative_prediction_per_target"] = negative_prediction_tables["per_target"]
    selected_test_prediction_type = (
        "projected"
        if "projected_predictions" in result["test_report"]
        else "raw"
    )
    selected_test_negative_summary = result["negative_prediction_summary"].loc[
        (result["negative_prediction_summary"]["split"] == "test")
        & (
            result["negative_prediction_summary"]["prediction_type"]
            == selected_test_prediction_type
        )
    ].iloc[0]
    training_context_artifacts = persist_classical_training_context(
        model_name,
        result,
        repo_root=repo_root,
        timestamp=training_timestamp,
    )
    result["notebook_training_context_artifacts"] = training_context_artifacts
    result["notebook_training_timestamp"] = training_timestamp

    print(f"{model_name} training complete.")
    print(
        "This benchmark run uses the same operational-plus-fractional inputs, the same fractional effluent targets, and the same train-test rows used by icsor."
    )
    print(
        "Externally collapsed measured-output metrics remain the direct comparison layer for the cross-model benchmark tables."
    )
    if projection_active:
        print("Fractional-space post-projection is active for this model because the Petersen null space is non-trivial.")
    else:
        print("Fractional-space post-projection is inactive for this model because the Petersen null space is trivial.")
        print("Projected outputs and fractional-space discrepancy summaries are therefore omitted below.")
    print(f"Saved model bundle: {result['artifact_paths']['model_bundle']}")
    print(f"Saved metrics summary: {result['artifact_paths']['metrics']}")
    print(f"Saved Optuna summary: {result['artifact_paths']['optuna']}")
    print(
        f"Saved notebook training context under timestamp: {training_context_artifacts['artifact_timestamp']}"
    )
    print(
        f"{model_name} {selected_test_prediction_type} test predictions are negative in "
        f"{int(selected_test_negative_summary['negative_predictions'])} of "
        f"{int(selected_test_negative_summary['total_predictions'])} predicted target values "
        f"({selected_test_negative_summary['negative_prediction_rate_pct']:.2f}%)."
    )

    record_training_table(
        f"{model_name} report metadata",
        "This table explains the reporting contract for the current benchmark model. Fractional-space outputs are native to training and inference, while externally collapsed measured-output metrics remain directly comparable with icsor.",
        result["test_report"]["report_metadata"],
    )
    record_training_table(
        f"{model_name} hyperparameters",
        "This table lists the hyperparameters used to fit the current benchmark model.",
        pd.DataFrame([result["best_hyperparameters"]]),
    )
    record_training_table(
        f"{model_name} train aggregate measured metrics",
        "This table reports the train-split externally collapsed measured-output metrics for the current benchmark model. These values are directly comparable with icsor.",
        result["train_report"]["aggregate_metrics"],
    )
    record_training_table(
        f"{model_name} test aggregate measured metrics",
        "This table reports the test-split externally collapsed measured-output metrics for the current benchmark model. These values are directly comparable with icsor.",
        result["test_report"]["aggregate_metrics"],
    )
    record_training_table(
        f"{model_name} test measured per-target metrics",
        "This table breaks down the current benchmark model's test metrics by the externally collapsed measured-output target labels.",
        result["test_report"]["per_target_metrics"],
    )
    if "fractional_aggregate_metrics" in result["train_report"]:
        record_training_table(
            f"{model_name} train aggregate fractional metrics",
            "This table reports the train-split native fractional-space metrics for the current benchmark model.",
            result["train_report"]["fractional_aggregate_metrics"],
        )
    if "fractional_aggregate_metrics" in result["test_report"]:
        record_training_table(
            f"{model_name} test aggregate fractional metrics",
            "This table reports the test-split native fractional-space metrics for the current benchmark model.",
            result["test_report"]["fractional_aggregate_metrics"],
        )
    if "fractional_per_target_metrics" in result["test_report"]:
        record_training_table(
            f"{model_name} test fractional per-target metrics",
            "This table breaks down the current benchmark model's test metrics by native ASM fractional target.",
            result["test_report"]["fractional_per_target_metrics"],
        )
    record_training_table(
        f"{model_name} negative prediction incidence summary",
        "This table summarizes how often the current benchmark model produces negative externally collapsed measured-output predictions on the train and test splits. When fractional-space post-projection is active, both raw and projected predictions are reported; otherwise only the raw predictions are shown.",
        result["negative_prediction_summary"],
    )
    record_training_table(
        f"{model_name} negative prediction incidence by target",
        "This table breaks down the negative-prediction incidence by externally collapsed measured-output target for the current benchmark model, including counts, rates, and the observed magnitude of the negative predictions for each available prediction type.",
        result["negative_prediction_per_target"],
    )
    if projection_active:
        record_training_table(
            f"{model_name} test diagnostic summary",
            "This table summarizes model-native diagnostics for the current benchmark model, including fractional-space constraint residuals and raw-to-projected adjustment magnitudes. These diagnostics are complementary to, but distinct from, the externally collapsed comparison metrics.",
            result["test_report"]["diagnostic_summary"],
        )
        record_training_table(
            f"{model_name} test constraint residual summary",
            "This table summarizes the sample-level fractional-space constraint residual norms on the current benchmark model's test split.",
            result["test_report"]["constraint_residuals"].describe().T,
        )

    return result

In [ ]:
classical_regressor_results = {}

for model_name in classical_regressor_specs:
    print(f"\n=== Running {model_name} ===")
    classical_regressor_results[model_name] = run_classical_regressor(
        model_name,
        use_optuna=USE_OPTUNA,
        persist_artifacts=True,
    )

list(classical_regressor_results)

## Foundational Tabular Models

This section benchmarks TabPFN and TabICL under the same fractional-output training contract used for the additional tabular regressors above. The models keep their own learned tabular inference machinery, but the notebook still evaluates them on the shared train-test split and records both the native fractional reports and the externally collapsed measured-output comparison tables.

In [ ]:
from src.models.ml import (
    load_tabicl_regressor_params,
    load_tabpfn_regressor_params,
    run_tabicl_regressor_pipeline,
    run_tabpfn_regressor_pipeline,
)
from src.models.ml.tabicl_regressor import build_tabicl_regressor_model
from src.models.ml.tabpfn_regressor import build_tabpfn_regressor_model

def load_tabpfn_regressor_params_ignore_limits():
    model_params = dict(load_tabpfn_regressor_params())
    training_defaults = dict(model_params["training_defaults"])
    training_defaults["ignore_pretraining_limits"] = True
    model_params["training_defaults"] = training_defaults
    return model_params

foundational_regressor_specs = {
    "tabpfn_regressor": {
        "load_params": load_tabpfn_regressor_params_ignore_limits,
        "build_model": build_tabpfn_regressor_model,
        "runner": run_tabpfn_regressor_pipeline,
    },
    "tabicl_regressor": {
        "load_params": load_tabicl_regressor_params,
        "build_model": build_tabicl_regressor_model,
        "runner": run_tabicl_regressor_pipeline,
    },
}

foundational_regressor_results = {}

for model_name in foundational_regressor_specs:
    print(f"\n=== Running {model_name} ===")
    foundational_regressor_results[model_name] = run_classical_regressor(
        model_name,
        use_optuna=USE_OPTUNA,
        persist_artifacts=True,
        spec_map=foundational_regressor_specs,
    )

if "classical_regressor_results" not in globals():
    classical_regressor_results = {}
classical_regressor_results.update(foundational_regressor_results)

list(foundational_regressor_results)


=== Running tabpfn_regressor ===


Training tabpfn_regressor:  40%|████      | 2/5 [00:16<00:25,  8.40s/stage, objective=fit, stage=evaluate_train]c:\Users\eggy\dissertation-files-final\pibre-model\.venv\Lib\site-packages\tabpfn\architectures\base\attention\full_attention.py:584: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  chunk_output = torch.nn.functional.scaled_dot_product_attention(
Training tabpfn_regressor:  60%|██████    | 3/5 [10:57<09:03, 271.82s/stage, objective=fit, stage=evaluate_test] error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded wi

tabpfn_regressor training complete.
This benchmark run uses the same operational-plus-fractional inputs, the same measured targets, and the same train-test rows used by icsor.
Measured-space post-projection is inactive for this model because the measured-output null space is trivial.
Projected outputs and measured-space discrepancy summaries are therefore omitted below.
Saved model bundle: C:\Users\eggy\dissertation-files-final\pibre-model\results\tabpfn_regressor\model_20260414_162720.pkl
Saved metrics summary: C:\Users\eggy\dissertation-files-final\pibre-model\results\tabpfn_regressor\metrics_20260414_162720.json
Saved Optuna summary: None
Saved notebook training context under timestamp: 20260414_162722
tabpfn_regressor raw test predictions are negative in 0 of 8000 predicted target values (0.00%).
tabpfn_regressor report metadata
This table explains the reporting contract for the current benchmark model. Measured-output metrics are directly comparable with icsor, and trivial measure

,native_prediction_space,comparison_target_space,constraint_space,direct_comparison_scope,diagnostic_scope,projection_active,constraint_status
0,measured,measured,measured,measured_output_metrics_only,projection_inactive_trivial_measured_null_space,False,inactive_trivial_null_space


tabpfn_regressor hyperparameters
This table lists the hyperparameters used to fit the current benchmark model.


...

tabpfn_regressor negative prediction incidence by target
This table breaks down the negative-prediction incidence by measured-output target for the current benchmark model, including counts, rates, and the observed magnitude of the negative predictions for each available prediction type.


,split,prediction_type,target,negative_predictions,negative_prediction_rate_pct,minimum_prediction,mean_negative_prediction,median_negative_prediction
0,test,raw,Out_TP,0,0.0,7.338379,NaN,NaN
1,test,raw,Out_TN,0,0.0,15.010536,NaN,NaN
2,test,raw,Out_TSS,0,0.0,195.874496,NaN,NaN
3,test,raw,Out_COD,0,0.0,218.055389,NaN,NaN
4,train,raw,Out_TP,0,0.0,5.986630,NaN,NaN
5,train,raw,Out_TN,0,0.0,14.168343,NaN,NaN
6,train,raw,Out_TSS,0,0.0,155.686066,NaN,NaN
7,train,raw,Out_COD,0,0.0,196.875549,NaN,NaN



=== Running tabicl_regressor ===


RuntimeError: Expected one of cpu, cuda, ipu, xpu, mkldnn, opengl, opencl, ideep, hip, ve, fpga, maia, xla, lazy, vulkan, mps, meta, hpu, mtia, privateuseone device type at start of device string: auto

error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))


## Analysis

Each code cell below runs the configurable dataset-size sweep for one model under the shared benchmark contract used by the main training sections. The boxplots compare measured-output metrics across train and test splits, while any additional diagnostic tables remain model-native and should not be interpreted as direct apples-to-apples ranking metrics across icsor and the tabular regressors.

When a tabular model does not expose the requested projected metric because measured-space projection is inactive, the notebook automatically falls back to the matching raw metric and reports that fallback in the cell output.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display as ipython_display

from src.utils.analysis import (
    build_notebook_table_recorder,
    load_latest_analysis_result,
    load_latest_classical_training_context,
    load_latest_icsor_training_context,
    persist_analysis_result_artifacts,
    run_model_dataset_size_analysis,
)
from src.utils.plot import persist_figure_artifacts, plot_train_test_metric_boxplots
from src.utils.simulation import make_simulation_timestamp


def _analysis_repo_root():
    return globals().get("repo_root")


def _slugify_artifact_name(value: str) -> str:
    resolved_value = "".join(
        character.lower() if character.isalnum() else "_"
        for character in str(value)
    )
    return "_".join(part for part in resolved_value.split("_") if part) or "artifact"


def resolve_analysis_metric(metric_frame, requested_metric: str, *, model_label: str) -> str:
    requested_metric_name = str(requested_metric)
    available_metrics = sorted(
        column_name
        for column_name in metric_frame.columns
        if column_name.startswith(("projected_", "raw_"))
    )

    if requested_metric_name in metric_frame.columns:
        return requested_metric_name

    fallback_metric_name = (
        requested_metric_name.replace("projected_", "raw_", 1)
        if requested_metric_name.startswith("projected_")
        else None
    )
    if fallback_metric_name and fallback_metric_name in metric_frame.columns:
        print(
            f"{model_label} analysis fallback: using '{fallback_metric_name}' because "
            f"'{requested_metric_name}' is unavailable for this result."
        )
        return fallback_metric_name

    raise KeyError(
        f"Requested metric '{requested_metric_name}' is unavailable for {model_label} analysis. "
        f"Available metrics: {', '.join(available_metrics)}."
    )


def report_analysis_metric_selection(
    metric_frame,
    requested_metric: str,
    *,
    model_label: str,
    ) -> str:
    effective_metric_name = resolve_analysis_metric(
        metric_frame,
        requested_metric,
        model_label=model_label,
    )
    print(f"Requested metric: {requested_metric}")
    print(f"Effective metric: {effective_metric_name}")
    return effective_metric_name


def resolve_icsor_analysis_hyperparameters(*, default_hyperparameters):
    if "icsor_result" in globals():
        return dict(icsor_result["best_hyperparameters"]), "the in-memory icsor training run"

    try:
        loaded_context = load_latest_icsor_training_context(repo_root=_analysis_repo_root())
    except FileNotFoundError:
        print("icsor analysis fallback: no saved training context was found, so the model defaults will be used.")
        return dict(default_hyperparameters), "the icsor training defaults"

    print(
        "icsor analysis fallback: loaded the latest saved training context from "
        f"{loaded_context['artifact_timestamp']}."
    )
    return dict(loaded_context["best_hyperparameters"]), (
        f"the saved icsor training context from {loaded_context['artifact_timestamp']}"
    )


def resolve_classical_analysis_hyperparameters(
    model_name: str,
    *,
    default_hyperparameters,
):
    classical_results = dict(globals().get("classical_regressor_results", {}))
    previous_result = classical_results.get(model_name)
    if previous_result is not None:
        return dict(previous_result["best_hyperparameters"]), f"the in-memory {model_name} training run"

    try:
        loaded_context = load_latest_classical_training_context(
            model_name,
            repo_root=_analysis_repo_root(),
        )
    except FileNotFoundError:
        print(
            f"{model_name} analysis fallback: no saved training context was found, so the model defaults will be used."
        )
        return dict(default_hyperparameters), f"the {model_name} training defaults"

    print(
        f"{model_name} analysis fallback: loaded the latest saved training context from "
        f"{loaded_context['artifact_timestamp']}."
    )
    return dict(loaded_context["best_hyperparameters"]), (
        f"the saved {model_name} training context from {loaded_context['artifact_timestamp']}"
    )


def load_analysis_result_from_memory_or_artifacts(
    model_name: str,
    *,
    result_key: str,
    model_label: str,
):
    if result_key in globals():
        return globals()[result_key]

    loaded_result = load_latest_analysis_result(
        model_name,
        repo_root=_analysis_repo_root(),
    )
    globals()[result_key] = loaded_result
    print(
        f"{model_label} comparison fallback: loaded the latest saved analysis bundle from "
        f"{loaded_result['artifact_timestamp']}."
    )
    return loaded_result


def run_standard_analysis(
    *,
    model_name: str,
    model_label: str,
    dataset,
    constraint_matrix,
    load_params,
    runner,
    target_columns,
    result_key: str,
    extra_runner_kwargs: dict | None = None,
    use_icsor_context: bool = False,
):
    if "analysis_metric" not in globals():
        raise NameError(
            "Run the first notebook cell before the analysis section so analysis_metric is defined."
        )

    requested_analysis_metric = str(globals()["analysis_metric"])
    analysis_overrides = dict(globals().get("analysis_overrides", {}))
    model_params = load_params()
    default_hyperparameters = dict(model_params["training_defaults"])
    if use_icsor_context:
        analysis_hyperparameters, hyperparameter_source = resolve_icsor_analysis_hyperparameters(
            default_hyperparameters=default_hyperparameters,
        )
    else:
        analysis_hyperparameters, hyperparameter_source = resolve_classical_analysis_hyperparameters(
            model_name,
            default_hyperparameters=default_hyperparameters,
        )

    resolved_extra_runner_kwargs = dict(extra_runner_kwargs or {})
    if "composition_matrix" in globals() and "composition_matrix" not in resolved_extra_runner_kwargs:
        resolved_extra_runner_kwargs["composition_matrix"] = composition_matrix
    if (
        "metadata" in globals()
        and "measured_output_columns" in metadata
        and "measured_output_columns" not in resolved_extra_runner_kwargs
    ):
        resolved_extra_runner_kwargs["measured_output_columns"] = list(metadata["measured_output_columns"])

    print(f"{model_label} analysis uses hyperparameters from {hyperparameter_source}.")
    analysis_result = run_model_dataset_size_analysis(
        model_name,
        dataset,
        constraint_matrix,
        runner,
        model_params=model_params,
        model_hyperparameters=analysis_hyperparameters,
        persist_artifacts=False,
        extra_runner_kwargs=resolved_extra_runner_kwargs,
        **analysis_overrides,
    )

    analysis_timestamp = make_simulation_timestamp()
    persisted_analysis_bundle = persist_analysis_result_artifacts(
        model_name,
        analysis_result,
        repo_root=_analysis_repo_root(),
        timestamp=analysis_timestamp,
    )
    record_analysis_table = build_notebook_table_recorder(
        f"analysis/{model_name}/summary",
        repo_root=_analysis_repo_root(),
        timestamp=analysis_timestamp,
        index=False,
    )
    effective_analysis_metric = report_analysis_metric_selection(
        analysis_result["per_target_metrics"],
        requested_analysis_metric,
        model_label=model_label,
    )

    print(f"{model_label} analysis complete.")
    print(f"Prediction tables returned: {len(analysis_result['prediction_tables'])}")
    print(
        f"Saved analysis bundle: {persisted_analysis_bundle['artifact_group']} "
        f"@ {persisted_analysis_bundle['artifact_timestamp']}"
    )
    record_analysis_table(
        f"{model_label} analysis configuration",
        f"This table lists the dataset-size sweep configuration used to evaluate {model_label} under the shared benchmark contract.",
        pd.DataFrame([analysis_result["analysis_config"]]),
    )
    record_analysis_table(
        f"{model_label} analysis run metadata preview",
        f"This table previews the run-level metadata for the {model_label} analysis sweep, including the sampled dataset size, repeat index, train size, test size, and run seed.",
        analysis_result["run_metadata"].head(),
    )

    for target_name in target_columns:
        figure, _ = plot_train_test_metric_boxplots(
            analysis_result["per_target_metrics"],
            metric_name=effective_analysis_metric,
            target_name=target_name,
            model_name=model_label,
        )
        persist_figure_artifacts(
            figure,
            f"analysis/{model_name}",
            f"{_slugify_artifact_name(target_name)}_{_slugify_artifact_name(effective_analysis_metric)}_boxplot",
            repo_root=_analysis_repo_root(),
            timestamp=analysis_timestamp,
        )
        ipython_display(figure)
        plt.close(figure)

    globals()[result_key] = analysis_result
    return analysis_result

In [ ]:
from src.models.ml import load_icsor_params, run_icsor_pipeline

icsor_analysis_result = run_standard_analysis(
    model_name="icsor",
    model_label="icsor",
    dataset=icsor_dataset,
    constraint_matrix=icsor_A_matrix,
    load_params=load_icsor_params,
    runner=run_icsor_pipeline,
    target_columns=icsor_dataset.targets.columns,
    result_key="icsor_analysis_result",
    extra_runner_kwargs={"composition_matrix": composition_matrix},
    use_icsor_context=True,
)

icsor_analysis_result

In [ ]:
from src.models.ml import load_xgboost_regressor_params, run_xgboost_regressor_pipeline

xgboost_analysis_result = run_standard_analysis(
    model_name="xgboost_regressor",
    model_label="XGBoost Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_xgboost_regressor_params,
    runner=run_xgboost_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="xgboost_analysis_result",
)

xgboost_analysis_result

In [ ]:
from src.models.ml import load_lightgbm_regressor_params, run_lightgbm_regressor_pipeline

lightgbm_analysis_result = run_standard_analysis(
    model_name="lightgbm_regressor",
    model_label="LightGBM Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_lightgbm_regressor_params,
    runner=run_lightgbm_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="lightgbm_analysis_result",
)

lightgbm_analysis_result

In [ ]:
from src.models.ml import load_catboost_regressor_params, run_catboost_regressor_pipeline

catboost_analysis_result = run_standard_analysis(
    model_name="catboost_regressor",
    model_label="CatBoost Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_catboost_regressor_params,
    runner=run_catboost_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="catboost_analysis_result",
)

catboost_analysis_result

In [ ]:
from src.models.ml import load_adaboost_regressor_params, run_adaboost_regressor_pipeline

adaboost_analysis_result = run_standard_analysis(
    model_name="adaboost_regressor",
    model_label="AdaBoost Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_adaboost_regressor_params,
    runner=run_adaboost_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="adaboost_analysis_result",
)

adaboost_analysis_result

In [ ]:
from src.models.ml import load_random_forest_regressor_params, run_random_forest_regressor_pipeline

random_forest_analysis_result = run_standard_analysis(
    model_name="random_forest_regressor",
    model_label="Random Forest Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_random_forest_regressor_params,
    runner=run_random_forest_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="random_forest_analysis_result",
)

random_forest_analysis_result

In [ ]:
from src.models.ml import load_svr_regressor_params, run_svr_regressor_pipeline

svr_analysis_result = run_standard_analysis(
    model_name="svr_regressor",
    model_label="SVR Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_svr_regressor_params,
    runner=run_svr_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="svr_analysis_result",
)

svr_analysis_result

In [ ]:
from src.models.ml import load_knn_regressor_params, run_knn_regressor_pipeline

knn_analysis_result = run_standard_analysis(
    model_name="knn_regressor",
    model_label="K-Nearest Neighbors Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_knn_regressor_params,
    runner=run_knn_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="knn_analysis_result",
)

knn_analysis_result

In [ ]:
from src.models.ml import load_pls_regressor_params, run_pls_regressor_pipeline

pls_analysis_result = run_standard_analysis(
    model_name="pls_regressor",
    model_label="Partial Least Squares Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_pls_regressor_params,
    runner=run_pls_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="pls_analysis_result",
)

pls_analysis_result

In [ ]:
from src.models.ml import load_ann_shallow_regressor_params, run_ann_shallow_regressor_pipeline

ann_shallow_analysis_result = run_standard_analysis(
    model_name="ann_shallow_regressor",
    model_label="ANN Shallow Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_ann_shallow_regressor_params,
    runner=run_ann_shallow_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="ann_shallow_analysis_result",
)

ann_shallow_analysis_result

In [ ]:
from src.models.ml import load_ann_medium_regressor_params, run_ann_medium_regressor_pipeline

ann_medium_analysis_result = run_standard_analysis(
    model_name="ann_medium_regressor",
    model_label="ANN Medium Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_ann_medium_regressor_params,
    runner=run_ann_medium_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="ann_medium_analysis_result",
)

ann_medium_analysis_result

In [ ]:
from src.models.ml import load_ann_deep_regressor_params, run_ann_deep_regressor_pipeline

ann_deep_analysis_result = run_standard_analysis(
    model_name="ann_deep_regressor",
    model_label="ANN Deep Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_ann_deep_regressor_params,
    runner=run_ann_deep_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="ann_deep_analysis_result",
)

ann_deep_analysis_result

In [ ]:
from src.models.ml import load_tabpfn_regressor_params, run_tabpfn_regressor_pipeline

tabpfn_analysis_result = run_standard_analysis(
    model_name="tabpfn_regressor",
    model_label="TabPFN Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_tabpfn_regressor_params,
    runner=run_tabpfn_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="tabpfn_analysis_result",
)

tabpfn_analysis_result

In [ ]:
from src.models.ml import load_tabicl_regressor_params, run_tabicl_regressor_pipeline

tabicl_analysis_result = run_standard_analysis(
    model_name="tabicl_regressor",
    model_label="TabICL Regressor",
    dataset=classical_benchmark_dataset,
    constraint_matrix=A_matrix,
    load_params=load_tabicl_regressor_params,
    runner=run_tabicl_regressor_pipeline,
    target_columns=classical_benchmark_dataset.targets.columns,
    result_key="tabicl_analysis_result",
)

tabicl_analysis_result

## Comprehensive Cross-Model Comparison

This section converts the per-model dataset-size sweeps above into one shared comparison workspace so every model is judged from the same evidence.

The comparison is organized around four complementary axes:

1. Primary predictive accuracy: effective test RMSE is the main ranking metric because it remains in measured-output units and penalizes large misses after any deployed post-processing.
2. Stability and generalization: repeated-resample means, interquartile ranges, and train-test gaps quantify robustness and overfitting instead of relying on one run.
3. Sample efficiency and target robustness: rankings are computed across training sizes and separately across each measured-output target, not only at the largest dataset size.
4. Physical and model-native diagnostics: negative prediction rates, raw-to-effective correction magnitudes, and available constraint residual summaries expose whether the models remain well-behaved physically.

In this section, `effective` means projected predictions when a model exposes them and raw predictions otherwise.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.utils.analysis import (
    build_notebook_table_recorder,
    collate_model_analysis_results,
    persist_named_table_artifacts,
)
from src.utils.plot import PIBRE_THEME_TOKENS, plot_metric_heatmap, plot_metric_summary_lines
from src.utils.simulation import make_simulation_timestamp

comparison_model_specs = [
    {"model_name": "icsor", "result_key": "icsor_analysis_result", "label": "icsor", "family": "Physics-informed"},
    {"model_name": "xgboost_regressor", "result_key": "xgboost_analysis_result", "label": "XGBoost Regressor", "family": "Classical"},
    {"model_name": "lightgbm_regressor", "result_key": "lightgbm_analysis_result", "label": "LightGBM Regressor", "family": "Classical"},
    {"model_name": "catboost_regressor", "result_key": "catboost_analysis_result", "label": "CatBoost Regressor", "family": "Classical"},
    {"model_name": "adaboost_regressor", "result_key": "adaboost_analysis_result", "label": "AdaBoost Regressor", "family": "Classical"},
    {"model_name": "random_forest_regressor", "result_key": "random_forest_analysis_result", "label": "Random Forest Regressor", "family": "Classical"},
    {"model_name": "svr_regressor", "result_key": "svr_analysis_result", "label": "SVR Regressor", "family": "Classical"},
    {"model_name": "knn_regressor", "result_key": "knn_analysis_result", "label": "K-Nearest Neighbors Regressor", "family": "Classical"},
    {"model_name": "pls_regressor", "result_key": "pls_analysis_result", "label": "Partial Least Squares Regressor", "family": "Classical"},
    {"model_name": "ann_shallow_regressor", "result_key": "ann_shallow_analysis_result", "label": "ANN Shallow Regressor", "family": "Classical"},
    {"model_name": "ann_medium_regressor", "result_key": "ann_medium_analysis_result", "label": "ANN Medium Regressor", "family": "Classical"},
    {"model_name": "ann_deep_regressor", "result_key": "ann_deep_analysis_result", "label": "ANN Deep Regressor", "family": "Classical"},
    {"model_name": "tabpfn_regressor", "result_key": "tabpfn_analysis_result", "label": "TabPFN Regressor", "family": "Foundational"},
    {"model_name": "tabicl_regressor", "result_key": "tabicl_analysis_result", "label": "TabICL Regressor", "family": "Foundational"},
]

comparison_results_by_model = {
    spec["model_name"]: load_analysis_result_from_memory_or_artifacts(
        spec["model_name"],
        result_key=spec["result_key"],
        model_label=spec["label"],
    )
    for spec in comparison_model_specs
}
comparison_model_labels = {
    spec["model_name"]: spec["label"]
    for spec in comparison_model_specs
}
comparison_model_families = {
    spec["model_name"]: spec["family"]
    for spec in comparison_model_specs
}

comparison_bundle = collate_model_analysis_results(
    comparison_results_by_model,
    model_labels=comparison_model_labels,
    model_families=comparison_model_families,
)
comparison_metric_basenames = ["RMSE", "MAE", "R2", "MAPE", "MSE"]
comparison_primary_metric = "effective_RMSE"
comparison_primary_split = "test"
comparison_base_model_columns = ["model_name", "model_label", "model_family", "model_order"]
comparison_analysis_configs = comparison_bundle["analysis_configs"].copy()
comparison_coverage = comparison_bundle["coverage"].copy()
comparison_run_metadata = comparison_bundle["run_metadata"].copy()
comparison_effective_aggregate_metrics = comparison_bundle["effective_aggregate_metrics"].copy()
comparison_per_target_metrics = comparison_bundle["per_target_metrics"].copy()
comparison_prediction_diagnostics = comparison_bundle["prediction_diagnostics"].copy()
comparison_prediction_target_diagnostics = comparison_bundle["p_target_diagnostics"].copy()
comparison_smallest_train_size = int(comparison_run_metadata["train_size"].min())
comparison_largest_train_size = int(comparison_run_metadata["train_size"].max())
comparison_target_labels = [
    str(target_name).removeprefix("Out_")
    for target_name in metadata["measured_output_columns"]
]
comparison_line_plot_kwargs = {
    "figure_size": (14.5, 8.0),
    "color_cycle": list(PIBRE_THEME_TOKENS["qualitative_cycle"]),
    "marker_cycle": list(PIBRE_THEME_TOKENS["line_marker_cycle"]),
    "linestyle_cycle": list(PIBRE_THEME_TOKENS["line_style_cycle"]),
    "legend_outside": True,
    "legend_location": "bottom",
    "legend_columns": 3,
}
comparison_core_timestamp = make_simulation_timestamp()
record_comparison_core_table = build_notebook_table_recorder(
    "comparison/core/summary",
    repo_root=repo_root,
    timestamp=comparison_core_timestamp,
    index=False,
)
persist_named_table_artifacts(
    "comparison/core",
    {
        "analysis_configs": comparison_analysis_configs,
        "coverage": comparison_coverage,
        "run_metadata": comparison_run_metadata,
        "effective_aggregate_metrics": comparison_effective_aggregate_metrics,
        "per_target_metrics": comparison_per_target_metrics,
        "prediction_diagnostics": comparison_prediction_diagnostics,
        "prediction_target_diagnostics": comparison_prediction_target_diagnostics,
    },
    repo_root=repo_root,
    timestamp=comparison_core_timestamp,
    default_index=False,
)


def resolve_heatmap_center_value(heatmap_frame):
    heatmap_values = np.asarray(heatmap_frame, dtype=float)
    finite_values = heatmap_values[np.isfinite(heatmap_values)]
    if finite_values.size == 0:
        return 0.0
    return float(0.5 * (finite_values.min() + finite_values.max()))


print("Comprehensive comparison bundle ready.")
print(f"Models included: {len(comparison_model_specs)}")
print(f"Primary ranking metric: {comparison_primary_metric}")
print(
    "Effective metrics use projected predictions when they are available and fall back to raw "
    "predictions when projection is not available for a model."
)
print(
    f"Training-size range covered by the sweep: {comparison_smallest_train_size} "
    f"to {comparison_largest_train_size}"
)
print(f"Saved comparison core tables under timestamp: {comparison_core_timestamp}")

record_comparison_core_table(
    "Comparison analysis configuration by model",
    "This table verifies that each model contributes the same dataset-size sweep configuration to the comprehensive comparison section.",
    comparison_analysis_configs,
)
record_comparison_core_table(
    "Comparison coverage and metric-source summary",
    "This table records the number of sweep runs, the train-test size range, the number of targets, and whether each model contributes projected or raw effective metrics to the direct cross-model comparison.",
    comparison_coverage,
)

In [ ]:
import numpy as np
import pandas as pd

from src.utils.analysis import (
    build_notebook_table_recorder,
    build_train_test_gap_summary,
    load_latest_named_table_artifacts,
    persist_named_table_artifacts,
    rank_metric_summary,
    summarize_metric_distribution,
)
from src.utils.simulation import make_simulation_timestamp

comparison_metric_basenames = globals().get("comparison_metric_basenames", ["RMSE", "MAE", "R2", "MAPE", "MSE"])
comparison_primary_metric = globals().get("comparison_primary_metric", "effective_RMSE")
comparison_primary_split = globals().get("comparison_primary_split", "test")
comparison_base_model_columns = globals().get(
    "comparison_base_model_columns",
    ["model_name", "model_label", "model_family", "model_order"],
)
comparison_core_required_artifacts = (
    "analysis_configs",
    "coverage",
    "run_metadata",
    "effective_aggregate_metrics",
    "per_target_metrics",
    "prediction_diagnostics",
    "prediction_target_diagnostics",
)

if "comparison_effective_aggregate_metrics" not in globals():
    loaded_comparison_core = load_latest_named_table_artifacts(
        "comparison/core",
        repo_root=repo_root,
        required_artifact_names=comparison_core_required_artifacts,
    )
    loaded_core_tables = loaded_comparison_core["tables"]
    comparison_analysis_configs = loaded_core_tables["analysis_configs"]
    comparison_coverage = loaded_core_tables["coverage"]
    comparison_run_metadata = loaded_core_tables["run_metadata"]
    comparison_effective_aggregate_metrics = loaded_core_tables["effective_aggregate_metrics"]
    comparison_per_target_metrics = loaded_core_tables["per_target_metrics"]
    comparison_prediction_diagnostics = loaded_core_tables["prediction_diagnostics"]
    comparison_prediction_target_diagnostics = loaded_core_tables["prediction_target_diagnostics"]
    comparison_smallest_train_size = int(comparison_run_metadata["train_size"].min())
    comparison_largest_train_size = int(comparison_run_metadata["train_size"].max())
    print(
        "Comparison derived fallback: loaded the latest saved comparison core bundle from "
        f"{loaded_comparison_core['artifact_timestamp']}."
    )

comparison_target_labels = globals().get("comparison_target_labels")
if comparison_target_labels is None:
    comparison_target_labels = (
        comparison_per_target_metrics["target"]
        .astype(str)
        .str.removeprefix("Out_")
        .drop_duplicates()
        .tolist()
    )


def merge_model_tables(table_list):
    non_empty_tables = [table.copy() for table in table_list if not table.empty]
    if not non_empty_tables:
        return pd.DataFrame(columns=comparison_base_model_columns)

    merged_table = non_empty_tables[0]
    for table in non_empty_tables[1:]:
        merged_table = merged_table.merge(
            table,
            on=comparison_base_model_columns,
            how="outer",
        )

    return merged_table.sort_values("model_order").reset_index(drop=True)


def build_curve_auc_table(summary_frame, *, metric_basename):
    rows = []
    for model_keys, group in summary_frame.groupby(comparison_base_model_columns, dropna=False):
        group = group.sort_values("train_size")
        x_values = group["train_size"].to_numpy(dtype=float)
        y_values = group["metric_mean"].to_numpy(dtype=float)
        normalized_curve_auc = (
            float(y_values[0])
            if len(group) == 1
            else float((np.trapezoid if hasattr(np, "trapezoid") else np.trapz)(y_values, x_values) / (x_values.max() - x_values.min()))
        )
        row = {
            column_name: model_value
            for column_name, model_value in zip(comparison_base_model_columns, model_keys)
        }
        row[f"{metric_basename}_curve_auc"] = normalized_curve_auc
        rows.append(row)

    return pd.DataFrame(rows)


def maybe_summarize_metric_distribution(metric_frame, *, metric_name, group_columns):
    if metric_name not in metric_frame.columns:
        return pd.DataFrame()

    non_null_frame = metric_frame.loc[metric_frame[metric_name].notna()].copy()
    if non_null_frame.empty:
        return pd.DataFrame()

    return summarize_metric_distribution(
        non_null_frame,
        metric_name=metric_name,
        group_columns=group_columns,
    )


comparison_test_aggregate_metrics = comparison_effective_aggregate_metrics.loc[
    comparison_effective_aggregate_metrics["split_name"] == comparison_primary_split
].copy()
comparison_test_per_target_metrics = comparison_per_target_metrics.loc[
    comparison_per_target_metrics["split_name"] == comparison_primary_split
].copy()

comparison_learning_curve_summaries = {}
comparison_target_rank_summaries = {}
comparison_largest_train_tables = []
comparison_average_rank_tables = []
comparison_curve_auc_tables = []

for metric_basename in comparison_metric_basenames:
    metric_name = f"effective_{metric_basename}"
    learning_curve_summary = summarize_metric_distribution(
        comparison_test_aggregate_metrics,
        metric_name=metric_name,
        group_columns=[*comparison_base_model_columns, "train_size"],
    )
    comparison_learning_curve_summaries[metric_name] = learning_curve_summary

    largest_metric_summary = learning_curve_summary.loc[
        learning_curve_summary["train_size"] == comparison_largest_train_size
    ].copy()
    largest_metric_ranked = rank_metric_summary(
        largest_metric_summary,
        group_columns=["train_size"],
        metric_name=metric_name,
    )
    comparison_largest_train_tables.append(
        largest_metric_ranked.loc[
            :,
            comparison_base_model_columns + ["metric_mean", "metric_std", "metric_rank"],
        ].rename(
            columns={
                "metric_mean": f"test_{metric_basename}_mean",
                "metric_std": f"test_{metric_basename}_std",
                "metric_rank": f"test_{metric_basename}_rank",
            }
        )
    )
    comparison_curve_auc_tables.append(
        build_curve_auc_table(learning_curve_summary, metric_basename=metric_basename)
    )

    target_summary = summarize_metric_distribution(
        comparison_test_per_target_metrics,
        metric_name=metric_name,
        group_columns=[*comparison_base_model_columns, "train_size", "target"],
    )
    target_ranked = rank_metric_summary(
        target_summary,
        group_columns=["train_size", "target"],
        metric_name=metric_name,
    )
    comparison_target_rank_summaries[metric_name] = target_ranked
    comparison_average_rank_tables.append(
        target_ranked.groupby(comparison_base_model_columns, dropna=False)["metric_rank"]
        .mean()
        .reset_index(name=f"{metric_basename}_average_rank")
    )

comparison_largest_train_size_leaderboard = merge_model_tables(comparison_largest_train_tables)
comparison_average_rank_table = merge_model_tables(comparison_average_rank_tables)
comparison_curve_auc_table = merge_model_tables(comparison_curve_auc_tables)

comparison_average_rank_columns = [
    f"{metric_basename}_average_rank"
    for metric_basename in comparison_metric_basenames
]
comparison_average_rank_table["overall_average_rank"] = comparison_average_rank_table[
    comparison_average_rank_columns
].mean(axis=1)
comparison_average_rank_table = comparison_average_rank_table.sort_values(
    ["overall_average_rank", "model_order"]
).reset_index(drop=True)
comparison_model_display_order = comparison_average_rank_table["model_label"].tolist()


def order_models(frame):
    ordered_frame = frame.copy()
    ordered_frame["model_label"] = pd.Categorical(
        ordered_frame["model_label"],
        categories=comparison_model_display_order,
        ordered=True,
    )
    ordered_frame = ordered_frame.sort_values(["model_label", "train_size"]).reset_index(drop=True)
    ordered_frame["model_label"] = ordered_frame["model_label"].astype(str)
    return ordered_frame


comparison_primary_train_test_summary = summarize_metric_distribution(
    comparison_effective_aggregate_metrics,
    metric_name=comparison_primary_metric,
    group_columns=[*comparison_base_model_columns, "split_name", "train_size"],
)
comparison_primary_gap_summary = build_train_test_gap_summary(
    comparison_primary_train_test_summary,
    group_columns=[*comparison_base_model_columns, "train_size"],
    metric_name=comparison_primary_metric,
)
comparison_primary_gap_largest = (
    comparison_primary_gap_summary.loc[
        comparison_primary_gap_summary["train_size"] == comparison_largest_train_size
    ]
    .rename(
        columns={
            "train_metric_mean": "train_effective_RMSE_mean",
            "test_metric_mean": "test_effective_RMSE_mean",
            "generalization_gap": "effective_RMSE_gap",
            "generalization_gap_pct": "effective_RMSE_gap_pct",
        }
    )
    .sort_values(["effective_RMSE_gap", "model_order"])
    .reset_index(drop=True)
)
comparison_primary_target_leaderboard = (
    comparison_target_rank_summaries[comparison_primary_metric]
    .loc[
        lambda frame: frame["train_size"] == comparison_largest_train_size,
        comparison_base_model_columns + ["target", "metric_mean", "metric_std", "metric_rank"],
    ]
    .rename(
        columns={
            "metric_mean": "effective_RMSE_mean",
            "metric_std": "effective_RMSE_std",
            "metric_rank": "effective_RMSE_rank",
        }
    )
    .sort_values(["target", "effective_RMSE_rank", "model_order"])
    .reset_index(drop=True)
)
comparison_primary_target_leaderboard["target"] = comparison_primary_target_leaderboard["target"].astype(str).str.removeprefix("Out_")

comparison_negative_rate_summary = summarize_metric_distribution(
    comparison_prediction_diagnostics.loc[
        comparison_prediction_diagnostics["split_name"] == comparison_primary_split
    ],
    metric_name="negative_prediction_rate_pct",
    group_columns=[*comparison_base_model_columns, "train_size"],
)
comparison_adjustment_summary = summarize_metric_distribution(
    comparison_prediction_diagnostics.loc[
        comparison_prediction_diagnostics["split_name"] == comparison_primary_split
    ],
    metric_name="raw_to_effective_adjustment_mean_l2",
    group_columns=[*comparison_base_model_columns, "train_size"],
)
comparison_constraint_summary = maybe_summarize_metric_distribution(
    comparison_prediction_diagnostics.loc[
        comparison_prediction_diagnostics["split_name"] == comparison_primary_split
    ],
    metric_name="effective_constraint_l2_mean",
    group_columns=[*comparison_base_model_columns, "train_size"],
)
comparison_negative_target_summary = summarize_metric_distribution(
    comparison_prediction_target_diagnostics.loc[
        comparison_prediction_target_diagnostics["split_name"] == comparison_primary_split
    ],
    metric_name="negative_prediction_rate_pct",
    group_columns=[*comparison_base_model_columns, "train_size", "target"],
)
comparison_negative_target_largest = comparison_negative_target_summary.loc[
    comparison_negative_target_summary["train_size"] == comparison_largest_train_size
].copy()
comparison_negative_target_largest["target"] = comparison_negative_target_largest["target"].astype(str).str.removeprefix("Out_")

comparison_diagnostic_tables = [
    comparison_negative_rate_summary.loc[
        comparison_negative_rate_summary["train_size"] == comparison_largest_train_size,
        comparison_base_model_columns + ["metric_mean", "metric_std"],
    ].rename(
        columns={
            "metric_mean": "negative_prediction_rate_pct_mean",
            "metric_std": "negative_prediction_rate_pct_std",
        }
    ),
    comparison_adjustment_summary.loc[
        comparison_adjustment_summary["train_size"] == comparison_largest_train_size,
        comparison_base_model_columns + ["metric_mean", "metric_std"],
    ].rename(
        columns={
            "metric_mean": "raw_to_effective_adjustment_l2_mean",
            "metric_std": "raw_to_effective_adjustment_l2_std",
        }
    ),
]
if not comparison_constraint_summary.empty:
    comparison_diagnostic_tables.append(
        comparison_constraint_summary.loc[
            comparison_constraint_summary["train_size"] == comparison_largest_train_size,
            comparison_base_model_columns + ["metric_mean", "metric_std"],
        ].rename(
            columns={
                "metric_mean": "effective_constraint_l2_mean",
                "metric_std": "effective_constraint_l2_std",
            }
        )
    )
comparison_diagnostic_summary = merge_model_tables(comparison_diagnostic_tables)
comparison_average_rank_heatmap = (
    comparison_average_rank_table
    .set_index("model_label")
    .loc[
        comparison_model_display_order,
        comparison_average_rank_columns + ["overall_average_rank"],
    ]
    .rename(
        columns={
            "RMSE_average_rank": "RMSE",
            "MAE_average_rank": "MAE",
            "R2_average_rank": "R2",
            "MAPE_average_rank": "MAPE",
            "MSE_average_rank": "MSE",
            "overall_average_rank": "Overall",
        }
    )
)
comparison_primary_target_heatmap = (
    comparison_primary_target_leaderboard
    .pivot(index="model_label", columns="target", values="effective_RMSE_mean")
    .reindex(index=comparison_model_display_order, columns=comparison_target_labels)
)
comparison_negative_target_heatmap = (
    comparison_negative_target_largest
    .pivot(index="model_label", columns="target", values="metric_mean")
    .reindex(index=comparison_model_display_order, columns=comparison_target_labels)
)

comparison_derived_timestamp = make_simulation_timestamp()
record_comparison_derived_table = build_notebook_table_recorder(
    "comparison/derived/summary",
    repo_root=repo_root,
    timestamp=comparison_derived_timestamp,
    index=False,
)
comparison_derived_tables = {
    "largest_train_size_leaderboard": comparison_largest_train_size_leaderboard,
    "average_rank_table": comparison_average_rank_table,
    "curve_auc_table": comparison_curve_auc_table,
    "primary_gap_largest": comparison_primary_gap_largest,
    "primary_target_leaderboard": comparison_primary_target_leaderboard,
    "diagnostic_summary": comparison_diagnostic_summary,
    "negative_rate_summary": comparison_negative_rate_summary,
    "adjustment_summary": comparison_adjustment_summary,
    "negative_target_largest": comparison_negative_target_largest,
    "primary_gap_summary": comparison_primary_gap_summary,
    "average_rank_heatmap": comparison_average_rank_heatmap.reset_index(),
    "primary_target_heatmap": comparison_primary_target_heatmap.reset_index(),
    "negative_target_heatmap": comparison_negative_target_heatmap.reset_index(),
    "model_display_order": pd.DataFrame({"model_label": comparison_model_display_order}),
    "comparison_meta": pd.DataFrame(
        [
            {
                "smallest_train_size": comparison_smallest_train_size,
                "largest_train_size": comparison_largest_train_size,
                "primary_metric": comparison_primary_metric,
                "primary_split": comparison_primary_split,
            }
        ]
    ),
    "learning_curve_summaries/effective_rmse": comparison_learning_curve_summaries["effective_RMSE"],
    "learning_curve_summaries/effective_mae": comparison_learning_curve_summaries["effective_MAE"],
    "learning_curve_summaries/effective_r2": comparison_learning_curve_summaries["effective_R2"],
    "learning_curve_summaries/effective_mape": comparison_learning_curve_summaries["effective_MAPE"],
}
if not comparison_constraint_summary.empty:
    comparison_derived_tables["constraint_summary"] = comparison_constraint_summary
persist_named_table_artifacts(
    "comparison/derived",
    comparison_derived_tables,
    repo_root=repo_root,
    timestamp=comparison_derived_timestamp,
    default_index=False,
)

print(
    f"All comparison tables below are evaluated at the largest training size "
    f"({comparison_largest_train_size} samples) unless stated otherwise."
)
print(f"Saved comparison derived tables under timestamp: {comparison_derived_timestamp}")

record_comparison_derived_table(
    "Largest-training-size aggregate leaderboard",
    "This table ranks the models at the largest training size using the aggregate effective test metrics. Effective RMSE is the primary ranking metric, while MAE, R2, MAPE, and MSE remain supporting metrics.",
    comparison_largest_train_size_leaderboard.round(4),
)
record_comparison_derived_table(
    "Average rank across metrics, targets, and training sizes",
    "This table averages each model's rank over all targets and all training sizes for every effective metric. Lower average ranks are better.",
    comparison_average_rank_table.round(4),
)
record_comparison_derived_table(
    "Learning-curve area summary",
    "This table summarizes the normalized area under each aggregate effective test learning curve. Lower areas are better for RMSE, MAE, MAPE, and MSE, while higher areas are better for R2.",
    comparison_curve_auc_table.round(4),
)
record_comparison_derived_table(
    "Largest-training-size generalization-gap summary",
    "This table reports the train-test effective RMSE gap at the largest training size. Positive gaps indicate that the train split outperforms the test split and therefore indicate stronger overfitting pressure.",
    comparison_primary_gap_largest.round(4),
)
record_comparison_derived_table(
    "Largest-training-size target leaderboard for effective RMSE",
    "This table ranks every model separately for each measured-output target using effective test RMSE at the largest training size.",
    comparison_primary_target_leaderboard.round(4),
)
record_comparison_derived_table(
    "Largest-training-size diagnostic summary",
    "This table compares negative prediction rates, raw-to-effective adjustment magnitudes, and the available model-native effective constraint residual summaries on the test split at the largest training size.",
    comparison_diagnostic_summary.round(4),
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display as ipython_display

from src.utils.analysis import load_latest_named_table_artifacts
from src.utils.plot import persist_figure_artifacts, plot_metric_heatmap, plot_metric_summary_lines
from src.utils.simulation import make_simulation_timestamp

comparison_metric_plot_specs = [
    ("effective_RMSE", "Effective test RMSE"),
    ("effective_MAE", "Effective test MAE"),
    ("effective_R2", "Effective test R2"),
    ("effective_MAPE", "Effective test MAPE"),
]
comparison_plot_required_artifacts = (
    "average_rank_heatmap",
    "model_display_order",
    "learning_curve_summaries/effective_rmse",
    "learning_curve_summaries/effective_mae",
    "learning_curve_summaries/effective_r2",
    "learning_curve_summaries/effective_mape",
)

if "comparison_learning_curve_summaries" not in globals() or "comparison_average_rank_heatmap" not in globals():
    loaded_comparison_derived = load_latest_named_table_artifacts(
        "comparison/derived",
        repo_root=repo_root,
        required_artifact_names=comparison_plot_required_artifacts,
    )
    loaded_plot_tables = loaded_comparison_derived["tables"]
    comparison_learning_curve_summaries = {
        "effective_RMSE": loaded_plot_tables["learning_curve_summaries/effective_rmse"],
        "effective_MAE": loaded_plot_tables["learning_curve_summaries/effective_mae"],
        "effective_R2": loaded_plot_tables["learning_curve_summaries/effective_r2"],
        "effective_MAPE": loaded_plot_tables["learning_curve_summaries/effective_mape"],
    }
    comparison_model_display_order = loaded_plot_tables["model_display_order"]["model_label"].tolist()
    comparison_average_rank_heatmap = loaded_plot_tables["average_rank_heatmap"].set_index("model_label")
    print(
        "Comparison learning-curve plot fallback: loaded the latest saved comparison derived bundle from "
        f"{loaded_comparison_derived['artifact_timestamp']}."
    )


def order_models(frame):
    ordered_frame = frame.copy()
    ordered_frame["model_label"] = pd.Categorical(
        ordered_frame["model_label"],
        categories=comparison_model_display_order,
        ordered=True,
    )
    ordered_frame = ordered_frame.sort_values(["model_label", "train_size"]).reset_index(drop=True)
    ordered_frame["model_label"] = ordered_frame["model_label"].astype(str)
    return ordered_frame


comparison_plot_timestamp = make_simulation_timestamp()
for metric_name, metric_label in comparison_metric_plot_specs:
    figure, _ = plot_metric_summary_lines(
        order_models(comparison_learning_curve_summaries[metric_name]),
        x_column="train_size",
        y_column="metric_mean",
        group_column="model_label",
        lower_column="metric_q25",
        upper_column="metric_q75",
        title=f"{metric_label} across training sizes",
        x_label="Number of training samples",
        y_label=metric_label,
        legend_title="Model",
        **comparison_line_plot_kwargs,
    )
    persist_figure_artifacts(
        figure,
        "comparison/plots",
        f"{metric_name.lower()}_learning_curve",
        repo_root=repo_root,
        timestamp=comparison_plot_timestamp,
    )
    ipython_display(figure)
    plt.close(figure)

figure, _ = plot_metric_heatmap(
    comparison_average_rank_heatmap,
    title="Average test rank across metrics, targets, and training sizes",
    x_label="Comparison metric",
    y_label="Model",
    colorbar_label="Average rank",
    center_value=0.5 * (len(comparison_model_display_order) + 1),
)
persist_figure_artifacts(
    figure,
    "comparison/plots",
    "average_rank_heatmap",
    repo_root=repo_root,
    timestamp=comparison_plot_timestamp,
)
ipython_display(figure)
plt.close(figure)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display as ipython_display

from src.utils.analysis import load_latest_named_table_artifacts
from src.utils.plot import persist_figure_artifacts, plot_metric_heatmap, plot_metric_summary_lines
from src.utils.simulation import make_simulation_timestamp

comparison_detail_required_artifacts = (
    "primary_target_heatmap",
    "negative_target_heatmap",
    "primary_gap_summary",
    "adjustment_summary",
    "comparison_meta",
    "model_display_order",
)

if "comparison_primary_target_heatmap" not in globals() or "comparison_primary_gap_summary" not in globals():
    loaded_comparison_derived = load_latest_named_table_artifacts(
        "comparison/derived",
        repo_root=repo_root,
        required_artifact_names=comparison_detail_required_artifacts,
    )
    loaded_detail_tables = loaded_comparison_derived["tables"]
    comparison_primary_target_heatmap = loaded_detail_tables["primary_target_heatmap"].set_index("model_label")
    comparison_negative_target_heatmap = loaded_detail_tables["negative_target_heatmap"].set_index("model_label")
    comparison_primary_gap_summary = loaded_detail_tables["primary_gap_summary"]
    comparison_adjustment_summary = loaded_detail_tables["adjustment_summary"]
    comparison_constraint_summary = loaded_detail_tables.get("constraint_summary", pd.DataFrame())
    comparison_model_display_order = loaded_detail_tables["model_display_order"]["model_label"].tolist()
    comparison_meta = loaded_detail_tables["comparison_meta"].iloc[0].to_dict()
    comparison_largest_train_size = int(comparison_meta["largest_train_size"])
    print(
        "Comparison detail plot fallback: loaded the latest saved comparison derived bundle from "
        f"{loaded_comparison_derived['artifact_timestamp']}."
    )


def resolve_heatmap_center_value(heatmap_frame):
    heatmap_values = heatmap_frame.to_numpy(dtype=float)
    finite_values = heatmap_values[pd.notna(heatmap_values)]
    if finite_values.size == 0:
        return 0.0
    return float(0.5 * (finite_values.min() + finite_values.max()))


def order_models(frame):
    ordered_frame = frame.copy()
    ordered_frame["model_label"] = pd.Categorical(
        ordered_frame["model_label"],
        categories=comparison_model_display_order,
        ordered=True,
    )
    ordered_frame = ordered_frame.sort_values(["model_label", "train_size"]).reset_index(drop=True)
    ordered_frame["model_label"] = ordered_frame["model_label"].astype(str)
    return ordered_frame


comparison_detail_plot_timestamp = make_simulation_timestamp()
figure, _ = plot_metric_heatmap(
    comparison_primary_target_heatmap,
    title=f"Effective RMSE by target at {comparison_largest_train_size} training samples",
    x_label="Measured target",
    y_label="Model",
    colorbar_label="Effective RMSE",
    center_value=resolve_heatmap_center_value(comparison_primary_target_heatmap),
)
persist_figure_artifacts(
    figure,
    "comparison/plots",
    "effective_rmse_by_target_heatmap",
    repo_root=repo_root,
    timestamp=comparison_detail_plot_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_metric_heatmap(
    comparison_negative_target_heatmap,
    title=f"Negative prediction rate by target at {comparison_largest_train_size} training samples",
    x_label="Measured target",
    y_label="Model",
    colorbar_label="Negative prediction rate (%)",
    center_value=resolve_heatmap_center_value(comparison_negative_target_heatmap),
)
persist_figure_artifacts(
    figure,
    "comparison/plots",
    "negative_prediction_rate_by_target_heatmap",
    repo_root=repo_root,
    timestamp=comparison_detail_plot_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_metric_summary_lines(
    order_models(comparison_primary_gap_summary),
    x_column="train_size",
    y_column="generalization_gap",
    group_column="model_label",
    title="Effective RMSE generalization gap across training sizes",
    x_label="Number of training samples",
    y_label="Train-test gap in effective RMSE",
    legend_title="Model",
    **comparison_line_plot_kwargs,
)
persist_figure_artifacts(
    figure,
    "comparison/plots",
    "effective_rmse_generalization_gap",
    repo_root=repo_root,
    timestamp=comparison_detail_plot_timestamp,
)
ipython_display(figure)
plt.close(figure)

figure, _ = plot_metric_summary_lines(
    order_models(comparison_adjustment_summary),
    x_column="train_size",
    y_column="metric_mean",
    group_column="model_label",
    lower_column="metric_q25",
    upper_column="metric_q75",
    title="Raw-to-effective adjustment magnitude across training sizes",
    x_label="Number of training samples",
    y_label="Mean raw-to-effective adjustment L2",
    legend_title="Model",
    **comparison_line_plot_kwargs,
)
persist_figure_artifacts(
    figure,
    "comparison/plots",
    "raw_to_effective_adjustment_learning_curve",
    repo_root=repo_root,
    timestamp=comparison_detail_plot_timestamp,
)
ipython_display(figure)
plt.close(figure)

if not comparison_constraint_summary.empty:
    figure, _ = plot_metric_summary_lines(
        order_models(comparison_constraint_summary),
        x_column="train_size",
        y_column="metric_mean",
        group_column="model_label",
        lower_column="metric_q25",
        upper_column="metric_q75",
        title="Model-native effective constraint residuals across training sizes",
        x_label="Number of training samples",
        y_label="Mean effective constraint L2",
        legend_title="Model",
        **comparison_line_plot_kwargs,
    )
    persist_figure_artifacts(
        figure,
        "comparison/plots",
        "effective_constraint_residual_learning_curve",
        repo_root=repo_root,
        timestamp=comparison_detail_plot_timestamp,
    )
    ipython_display(figure)
    plt.close(figure)